# Q1-DIR-Q27-PROXY48 v1.2 — Hardened Direct Baseline / Interface Freeze
## ARC-AGI-2 | 4× NVIDIA L4 | Offline Kaggle | Qwen3.8-27B-FP8 TP=4

This revision keeps the **Q1 scientific intervention unchanged**: Direct Qwen3.8-27B, Thinking OFF vs Thinking XHIGH, frozen Proxy-48, same prompts, same sampling settings, same production decode budgets.

### v1.2 hardening after source + SMOKE re-audit

The following are **local research-hygiene fixes**, not new solver mechanisms:

1. **Resume contract fingerprinting** — a checkpoint is reusable only when task, rendered prompt SHA, model artifact fingerprint, runtime stack, chat template, engine config, sampling config/seed, profile and contract version match.
2. **Exact coverage sets** — completion is based on the exact compatible active-task set, not only a count; missing/foreign/stale tasks are reported.
3. **`no_output` is failed/incomplete** — an empty vLLM request output no longer gets `status="ok"` and therefore cannot satisfy resume/coverage.
4. **Provenance completeness** — raw rows and candidate registry keep run ID, record ID, raw seed, prompt/config/model fingerprints and source run provenance.
5. **PREFLIGHT/production dependency parity** — offline vLLM 0.19.0 dependency resolution is enforced before tokenizer rendering even in PREFLIGHT; PREFLIGHT still does not import vLLM/CUDA or allocate the 27B model.
6. **Truncation-safe parser from v1.1 retained** — length-truncated reasoning cannot donate an arbitrary intermediate ARC grid as the final candidate.

### Validation policy

Because checkpoint/coverage logic changed, this notebook intentionally defaults to:

```python
RUN_PROFILE = "PREFLIGHT"
```

Run PREFLIGHT once. If it passes, change only that line to `RUN_PROFILE="PROXY48"`.
The previously supplied SMOKE_GPU run already passed the unchanged 4×L4/TP4 model-load path; repeat SMOKE only if the environment/runtime gate changes or PREFLIGHT reports stack drift.


In [1]:
# =====================================================================================
# CELL 1 — Q1 CONFIG + OFFLINE ENVIRONMENT SETUP + FROZEN SOURCE CONTRACT
# Must run before importing torch/vLLM.
# =====================================================================================
import os
import sys
import json
import math
import time
import random
import hashlib
import zipfile
import shutil
import traceback
import importlib.metadata
from pathlib import Path
from datetime import datetime, timezone
from collections import deque

SEED = 42
random.seed(SEED)
EXPERIMENT_ID = "Q1-DIR-Q27-PROXY48-v1.2"
GENERATION_CONTRACT_VERSION = "q1-dir-q27-generation-v1.2"
PARSER_CONTRACT_VERSION = "q1-tolerant-grid-parser-v1.1-truncation-guard"
PROMPT_CONTRACT_VERSION = "q1-direct-minimal-json-v1"

# -------------------------------------------------------------------------------------
# FROZEN SOURCE AUDIT
# -------------------------------------------------------------------------------------
SOURCE_FILE_SHA256 = {
    "qwen_exp1_base": "07d7c7a7ca4567a6118ae8cef00b10277aa37d5ada8698afb66afeb43c88aeb1",
    "qwen_runtime_benchmark": "7d91965196ec3309bf4e3f16f90efcaf2067ee8575d245c8eab9eb100f204b78",
    "arc2_advanced_eda": "1a9384d4b2e01fc15cddad95dee1d6eb5a633b6ae2d041d3d9ae97ffd1879f64",
    "nvarc_31_experiment_diagnostics": "69a64354b6b1c696af1f1c7a68984275d4db9b7eef5f2a382f879dc862d7c978",
    "portability_notes_v10": "c2dd4b45e3697d298d8f814b4087cee742ef4f8632f0eac318ba16cb8f8abda8",
    "experiment_master_plan": "8ca1c34c2c3d864c4b8e04fadf64465cbf468a09b3647c690f7306a07daf0c42"
}

PROXY48_TASK_IDS = [
    "0934a4d8",
    "20a9e565",
    "4c7dc4dd",
    "89565ca0",
    "136b0064",
    "291dc1e1",
    "16b78196",
    "271d71e2",
    "28a6681f",
    "2b83f449",
    "2c181942",
    "31f7f899",
    "35ab12c3",
    "3dc255db",
    "409aa875",
    "4c416de3",
    "64efde09",
    "6e453dd6",
    "7491f3cf",
    "7b0280bc",
    "7b80bb43",
    "7c66cb00",
    "1818057f",
    "332f06d7",
    "53fb4810",
    "7666fa5d",
    "da515329",
    "8f3a5a89",
    "db695cfb",
    "97d7923e",
    "de809cff",
    "c7f57c3e",
    "cb2d8a2c",
    "58490d8a",
    "dbff022c",
    "5961cc34",
    "80a900e0",
    "269e22fb",
    "8b9c3697",
    "e376de54",
    "b99e7126",
    "c4d067a0",
    "135a2760",
    "4a21e3da",
    "71e489b6",
    "800d221b",
    "dd6b8c4b",
    "36a08778"
]
PROXY48_ORDERED_SHA256 = hashlib.sha256("\n".join(PROXY48_TASK_IDS).encode("utf-8")).hexdigest()
PROXY48_EXPECTED_SHA256 = "8f432c60a35091dcda2b890574b1f4d3c03aaf48a5b3b7f96b4e88c35d359134"
PROXY48_SOURCE_COUNTS = {
    "total": 48,
    "tier1": 32,
    "tier2": 12,
    "tier3": 4,
    "easy_static": 16,
    "rare": 30,
    "multi_query": 8,
    "dynamic": 8,
    "hard_tier1": 2,
    "high_ratio": 5,
    "extreme_ratio": 4,
}

if len(PROXY48_TASK_IDS) != 48 or len(set(PROXY48_TASK_IDS)) != 48:
    raise RuntimeError("Frozen Proxy-48 must contain exactly 48 unique task IDs.")
if PROXY48_ORDERED_SHA256 != PROXY48_EXPECTED_SHA256:
    raise RuntimeError("Frozen Proxy-48 fingerprint mismatch. Refusing to run a changed suite.")

# NVARC notebook diagnostic anchors. These are SOURCE diagnostics only, never Q1 results.
NVARC_DIAGNOSTIC_ANCHORS = {
    "outputs_analyzed": 172,
    "gt_present_in_pool": 55,
    "gt_pool_coverage": 55/172,
    "gt_selected_by_current_dual": 45,
    "current_dual_score_reported": 33.50,
    "oracle_score_reported": 41.17,
    "avg_raw_duplicate_ratio": 0.378,
    "scorer_rank_correlation_on_gt_present": 0.961,
    "purpose": "Motivate complementary candidate coverage and provenance-aware dedup; not used to tune Q1 predictions.",
}

# =====================================================================================
# ONE-LINE RUN SWITCH
# =====================================================================================
RUN_PROFILE = "PROXY48"   # "PREFLIGHT" | "SMOKE_GPU" | "PROXY48"
RESET_PROXY48_OUTPUTS = False

PROFILE_TABLE = {
    "PREFLIGHT": {
        "use_model": False,
        "task_set": "PROXY48",
        "active_tasks": 48,
        "off_max_tokens": 4096,
        "xhigh_max_tokens": 12288,
        "off_batch_size": 16,
        "xhigh_batch_size": 8,
        "context_probe_xhigh_tokens": 12288,
        "bundle_name": "Q1_DIR_Q27_PREFLIGHT_BUNDLE.zip",
        "reset_outputs": True,
        "accuracy_is_real": False,
    },
    "SMOKE_GPU": {
        "use_model": True,
        "task_set": "SMOKE16",
        "active_tasks": 16,
        "off_max_tokens": 512,
        "xhigh_max_tokens": 1024,
        "off_batch_size": 16,
        "xhigh_batch_size": 8,
        # Stress the real PROXY48 context/KV-cache request while keeping decode short.
        "context_probe_xhigh_tokens": 12288,
        "bundle_name": "Q1_DIR_Q27_SMOKE_GPU_BUNDLE.zip",
        "reset_outputs": True,
        "accuracy_is_real": False,  # runtime-only: 512/1024 decode caps intentionally truncate reasoning
    },
    "PROXY48": {
        "use_model": True,
        "task_set": "PROXY48",
        "active_tasks": 48,
        "off_max_tokens": 4096,
        "xhigh_max_tokens": 12288,
        "off_batch_size": 16,
        "xhigh_batch_size": 8,
        "context_probe_xhigh_tokens": 12288,
        "bundle_name": "Q1_DIR_Q27_PROXY48_BUNDLE.zip",
        "reset_outputs": RESET_PROXY48_OUTPUTS,
        "accuracy_is_real": True,
    },
}

RUN_PROFILE = str(RUN_PROFILE).strip().upper()
if RUN_PROFILE not in PROFILE_TABLE:
    raise ValueError(f"RUN_PROFILE={RUN_PROFILE!r} invalid; choose {list(PROFILE_TABLE)}")

PROFILE = PROFILE_TABLE[RUN_PROFILE]
USE_MODEL = bool(PROFILE["use_model"])
TASK_SET = str(PROFILE["task_set"])
ACTIVE_TASK_LIMIT = int(PROFILE["active_tasks"])
OFF_MAX_TOKENS = int(PROFILE["off_max_tokens"])
XHIGH_MAX_TOKENS = int(PROFILE["xhigh_max_tokens"])
OFF_BATCH_SIZE = int(PROFILE["off_batch_size"])
XHIGH_BATCH_SIZE = int(PROFILE["xhigh_batch_size"])
CONTEXT_PROBE_XHIGH_TOKENS = int(PROFILE["context_probe_xhigh_tokens"])
BUNDLE_NAME = str(PROFILE["bundle_name"])
ACCURACY_IS_REAL = bool(PROFILE["accuracy_is_real"])

MODEL_PATH = Path(
    "/kaggle/input/models/foysalemonshanto/"
    "qwen3-8-27b-fp8-repacked-v1/"
    "pytorch/hf-fp8/1"
)
WHEELHOUSE = Path(
    "/kaggle/input/datasets/driessmit1/"
    "arc3-vllm-h100-wheelhouse-v3"
)
ARC_ROOT = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2")

TRAIN_CHALLENGES_PATH = ARC_ROOT / "arc-agi_training_challenges.json"
TRAIN_SOLUTIONS_PATH  = ARC_ROOT / "arc-agi_training_solutions.json"
EVAL_CHALLENGES_PATH  = ARC_ROOT / "arc-agi_evaluation_challenges.json"
EVAL_SOLUTIONS_PATH   = ARC_ROOT / "arc-agi_evaluation_solutions.json"
TEST_CHALLENGES_PATH  = ARC_ROOT / "arc-agi_test_challenges.json"
SAMPLE_SUBMISSION_PATH = ARC_ROOT / "sample_submission.json"

BASE_WORK_ROOT = Path("/kaggle/working/q1_dir_q27_proxy48")
WORK_ROOT = BASE_WORK_ROOT / RUN_PROFILE.lower()
if PROFILE["reset_outputs"] and WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
NOTEBOOK_WALL_START = time.perf_counter()
TOTAL_RUNTIME_SEC = 12 * 60 * 60
SAFETY_RESERVE_SEC = 30 * 60

# Validated 4×L4 vLLM settings from the prior Qwen notebooks. DO NOT casually change in Q1.
MAX_NUM_BATCHED_TOKENS = 32768
MAX_NUM_SEQS = 16
GPU_MEMORY_UTILIZATION = 0.90

# EXP-0 propensity diagnostic is retained but remains secondary/non-official.
WEIGHT_CLIP_LOW = 0.20
WEIGHT_CLIP_HIGH = 5.00
N_PROPENSITY_FOLDS = 5
HIGH_RATIO_THRESHOLD = 10.0
LARGE_TEST_INPUT_CELLS = 300
MANY_COLORS_THRESHOLD = 7
LOW_BG_THRESHOLD = 0.30

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

print("=" * 116)
print(EXPERIMENT_ID, "| ARC-AGI-2 × Qwen3.8-27B-FP8 | OFF vs XHIGH")
print("=" * 116)
print("RUN_ID:", RUN_ID)
print("RUN_PROFILE:", RUN_PROFILE, "| TASK_SET:", TASK_SET)
print("USE_MODEL:", USE_MODEL)
print("ACTIVE_TASK_LIMIT:", ACTIVE_TASK_LIMIT)
print("OFF / XHIGH token caps:", OFF_MAX_TOKENS, "/", XHIGH_MAX_TOKENS)
print("OFF / XHIGH batch sizes:", OFF_BATCH_SIZE, "/", XHIGH_BATCH_SIZE)
print("Proxy-48 SHA256:", PROXY48_ORDERED_SHA256)
print("WORK_ROOT:", WORK_ROOT)

required = [
    MODEL_PATH, WHEELHOUSE, ARC_ROOT,
    TRAIN_CHALLENGES_PATH, TRAIN_SOLUTIONS_PATH,
    EVAL_CHALLENGES_PATH, EVAL_SOLUTIONS_PATH,
    TEST_CHALLENGES_PATH, SAMPLE_SUBMISSION_PATH,
]
for p in required:
    if not p.exists():
        raise FileNotFoundError(f"Required Kaggle input is missing: {p}")

config_path = MODEL_PATH / "config.json"
index_path = MODEL_PATH / "model.safetensors.index.json"
if not config_path.exists() or not index_path.exists():
    raise FileNotFoundError("Checkpoint config/index is incomplete.")

with open(config_path, "r", encoding="utf-8") as f:
    model_cfg = json.load(f)
qcfg = model_cfg.get("quantization_config", {})
print("Architecture :", model_cfg.get("architectures"))
print("Model type   :", model_cfg.get("model_type"))
print("Quantization :", qcfg.get("quant_method"), qcfg.get("fmt"), qcfg.get("activation_scheme"))
if model_cfg.get("model_type") != "qwen3_5":
    raise RuntimeError(f"Unexpected model_type={model_cfg.get('model_type')}")
if qcfg.get("quant_method") != "fp8" or qcfg.get("fmt") != "e4m3":
    raise RuntimeError(f"Unexpected quantization_config={qcfg}")

def pkg_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

def stable_json_sha256(obj):
    payload = json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def small_file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# Exact package stack observed in the successful 4×L4 SMOKE run.
EXPECTED_RUNTIME_PACKAGES = {
    "vllm": "0.19.0",
    "transformers": "4.57.6",
    "tokenizers": "0.22.2",
    "huggingface-hub": "0.36.2",
}

def offline_pip_install(requirements):
    if not requirements:
        return
    import subprocess
    print("Installing from offline wheelhouse:", requirements)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--no-index", "--no-cache-dir",
        "--find-links", str(WHEELHOUSE),
        *requirements,
    ])

before_stack = {name: pkg_version(name) for name in EXPECTED_RUNTIME_PACKAGES}
print("Runtime package stack before setup:", before_stack)
print("Torch distribution:", pkg_version("torch"))

_loaded_module_names = {
    "vllm": "vllm",
    "transformers": "transformers",
    "tokenizers": "tokenizers",
    "huggingface-hub": "huggingface_hub",
}
for _pkg, _module in _loaded_module_names.items():
    if before_stack.get(_pkg) != EXPECTED_RUNTIME_PACKAGES[_pkg] and _module in sys.modules:
        raise RuntimeError(
            f"{_pkg} is already imported with version {before_stack.get(_pkg)!r}, "
            f"but v1.2 requires {EXPECTED_RUNTIME_PACKAGES[_pkg]!r}. "
            "Restart the Kaggle session, then Run All."
        )

# PREFLIGHT must render with the same tokenizer/Transformers stack as GPU production.
# Installing the package does NOT import vLLM/CUDA and does NOT allocate the 27B model.
if before_stack["vllm"] != EXPECTED_RUNTIME_PACKAGES["vllm"]:
    offline_pip_install([f"vllm=={EXPECTED_RUNTIME_PACKAGES['vllm']}"])
else:
    repair = []
    for package in ["transformers", "tokenizers", "huggingface-hub"]:
        if before_stack[package] != EXPECTED_RUNTIME_PACKAGES[package]:
            module_name = package.replace("-", "_")
            if module_name in sys.modules or (package == "huggingface-hub" and "huggingface_hub" in sys.modules):
                raise RuntimeError(
                    f"{package} is already imported with version {before_stack[package]!r}; "
                    "restart the Kaggle session before repairing the runtime stack."
                )
            repair.append(f"{package}=={EXPECTED_RUNTIME_PACKAGES[package]}")
    offline_pip_install(repair)

after_stack = {name: pkg_version(name) for name in EXPECTED_RUNTIME_PACKAGES}
print("Runtime package stack after setup :", after_stack)
for package, wanted in EXPECTED_RUNTIME_PACKAGES.items():
    if after_stack.get(package) != wanted:
        raise RuntimeError(
            f"Runtime parity failure: {package}={after_stack.get(package)!r}, expected {wanted!r}."
        )

print("Torch distribution:", pkg_version("torch"))
print("Triton            :", pkg_version("triton"))
print("Dependency parity PASS. No model loaded yet.")


Q1-DIR-Q27-PROXY48-v1.2 | ARC-AGI-2 × Qwen3.8-27B-FP8 | OFF vs XHIGH
RUN_ID: 20260913T134749Z
RUN_PROFILE: PROXY48 | TASK_SET: PROXY48
USE_MODEL: True
ACTIVE_TASK_LIMIT: 48
OFF / XHIGH token caps: 4096 / 12288
OFF / XHIGH batch sizes: 16 / 8
Proxy-48 SHA256: 8f432c60a35091dcda2b890574b1f4d3c03aaf48a5b3b7f96b4e88c35d359134
WORK_ROOT: /kaggle/working/q1_dir_q27_proxy48/proxy48
Architecture : ['Qwen3_5ForConditionalGeneration']
Model type   : qwen3_5
Quantization : fp8 e4m3 dynamic
Runtime package stack before setup: {'vllm': None, 'transformers': '5.0.0', 'tokenizers': '0.22.2', 'huggingface-hub': '1.11.0'}
Torch distribution: 2.10.0+cu128
Installing from offline wheelhouse: ['vllm==0.19.0']
Looking in links: /kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3
Processing /kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3/vllm-0.19.0-cp38-abi3-manylinux_2_31_x86_64.whl
Processing /kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3/transformers-4.57.6-py3-

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.41.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.41.1 which is incompatible.


In [2]:

# =====================================================================================
# CELL 2 — LOAD ARC DATA
# =====================================================================================
import numpy as np
import pandas as pd
from IPython.display import display

np.random.seed(SEED)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_challenges = load_json(TRAIN_CHALLENGES_PATH)
train_solutions  = load_json(TRAIN_SOLUTIONS_PATH)
eval_challenges  = load_json(EVAL_CHALLENGES_PATH)
eval_solutions   = load_json(EVAL_SOLUTIONS_PATH)
test_challenges  = load_json(TEST_CHALLENGES_PATH)
sample_submission = load_json(SAMPLE_SUBMISSION_PATH)

assert set(eval_challenges) == set(eval_solutions)
assert set(test_challenges) == set(sample_submission)
missing_proxy_ids = [k for k in PROXY48_TASK_IDS if k not in eval_challenges or k not in eval_solutions]
if missing_proxy_ids:
    raise RuntimeError(f"Frozen Proxy-48 IDs missing from public Evaluation data: {missing_proxy_ids}")

print("Training tasks       :", len(train_challenges))
print("Evaluation tasks     :", len(eval_challenges))
print("Evaluation outputs   :", sum(len(v) for v in eval_solutions.values()))
print("Test tasks           :", len(test_challenges))
print("Visible test queries :", sum(len(v["test"]) for v in test_challenges.values()))


Training tasks       : 1000
Evaluation tasks     : 120
Evaluation outputs   : 172
Test tasks           : 240
Visible test queries : 259


In [3]:

# =====================================================================================
# CELL 3 — EXP-0 STRUCTURAL FEATURES + TEST-RELEVANCE WEIGHTS
# Recomputed here so EXP-1 is self-contained.
# =====================================================================================
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, log_loss

def arr(grid):
    return np.asarray(grid, dtype=np.int16)

def is_rectangular_grid(grid, max_hw=30):
    if not isinstance(grid, list) or len(grid) < 1 or len(grid) > max_hw:
        return False
    if not all(isinstance(row, list) for row in grid):
        return False
    widths = [len(row) for row in grid]
    if min(widths, default=0) < 1 or max(widths, default=0) > max_hw or len(set(widths)) != 1:
        return False
    for row in grid:
        for x in row:
            if isinstance(x, bool) or not isinstance(x, (int, np.integer)):
                return False
            if int(x) < 0 or int(x) > 9:
                return False
    return True

def grid_basic_stats(grid):
    a = arr(grid)
    vals, counts = np.unique(a, return_counts=True)
    probs = counts / counts.sum()
    entropy = float(-(probs * np.log(probs + 1e-12)).sum())
    return {
        "h": int(a.shape[0]),
        "w": int(a.shape[1]),
        "cells": int(a.size),
        "n_colors": int(len(vals)),
        "bg0_density": float(np.mean(a == 0)),
        "color_entropy": entropy,
    }

def colored_components_count(grid):
    # Diagnostic only: 4-connected same-color non-zero components.
    a = arr(grid)
    h, w = a.shape
    seen = np.zeros((h, w), dtype=bool)
    n = 0
    sizes = []
    for r in range(h):
        for c in range(w):
            if seen[r, c] or a[r, c] == 0:
                continue
            color = a[r, c]
            q = deque([(r, c)])
            seen[r, c] = True
            size = 0
            while q:
                rr, cc = q.popleft()
                size += 1
                for dr, dc in ((1,0),(-1,0),(0,1),(0,-1)):
                    nr, nc = rr+dr, cc+dc
                    if 0 <= nr < h and 0 <= nc < w and not seen[nr, nc] and a[nr, nc] == color:
                        seen[nr, nc] = True
                        q.append((nr, nc))
            n += 1
            sizes.append(size)
    return n, (float(np.mean(sizes)) if sizes else 0.0)

def summarize(values, prefix):
    x = np.asarray(values, dtype=float)
    if len(x) == 0:
        return {
            f"{prefix}_mean": 0.0, f"{prefix}_std": 0.0,
            f"{prefix}_min": 0.0, f"{prefix}_max": 0.0,
        }
    return {
        f"{prefix}_mean": float(x.mean()),
        f"{prefix}_std": float(x.std()),
        f"{prefix}_min": float(x.min()),
        f"{prefix}_max": float(x.max()),
    }

def task_features(task_id, task, split):
    train = task["train"]
    tests = task["test"]

    tr_in_stats, tr_out_stats, te_in_stats = [], [], []
    tr_in_components, tr_out_components, te_components = [], [], []
    tr_in_component_sizes, tr_out_component_sizes, te_component_sizes = [], [], []
    ratios, h_ratios, w_ratios, same_shape = [], [], [], []
    all_visible_colors = set()

    for ex in train:
        si = grid_basic_stats(ex["input"])
        so = grid_basic_stats(ex["output"])
        tr_in_stats.append(si); tr_out_stats.append(so)
        ci, csi = colored_components_count(ex["input"])
        co, cso = colored_components_count(ex["output"])
        tr_in_components.append(ci); tr_out_components.append(co)
        tr_in_component_sizes.append(csi); tr_out_component_sizes.append(cso)
        ratios.append(so["cells"] / max(si["cells"], 1))
        h_ratios.append(so["h"] / max(si["h"], 1))
        w_ratios.append(so["w"] / max(si["w"], 1))
        same_shape.append(int((si["h"], si["w"]) == (so["h"], so["w"])))
        all_visible_colors.update(np.unique(arr(ex["input"])).tolist())
        all_visible_colors.update(np.unique(arr(ex["output"])).tolist())

    for ex in tests:
        st = grid_basic_stats(ex["input"])
        te_in_stats.append(st)
        ct, cst = colored_components_count(ex["input"])
        te_components.append(ct); te_component_sizes.append(cst)
        all_visible_colors.update(np.unique(arr(ex["input"])).tolist())

    feat = {
        "task_id": task_id,
        "split": split,
        "n_train": len(train),
        "n_test_queries": len(tests),
        "visible_n_colors": len(all_visible_colors),
        "same_shape_frac": float(np.mean(same_shape)),
        "dynamic_any": int(any(x == 0 for x in same_shape)),
        "multi_query": int(len(tests) > 1),
    }

    for key in ["h", "w", "cells", "n_colors", "bg0_density", "color_entropy"]:
        feat.update(summarize([x[key] for x in tr_in_stats], f"tr_in_{key}"))
        feat.update(summarize([x[key] for x in tr_out_stats], f"tr_out_{key}"))
        feat.update(summarize([x[key] for x in te_in_stats], f"te_in_{key}"))

    feat.update(summarize(ratios, "area_ratio"))
    feat.update(summarize(h_ratios, "h_ratio"))
    feat.update(summarize(w_ratios, "w_ratio"))
    feat.update(summarize(tr_in_components, "tr_in_components"))
    feat.update(summarize(tr_out_components, "tr_out_components"))
    feat.update(summarize(te_components, "te_in_components"))
    feat.update(summarize(tr_in_component_sizes, "tr_in_component_size"))
    feat.update(summarize(tr_out_component_sizes, "tr_out_component_size"))
    feat.update(summarize(te_component_sizes, "te_in_component_size"))

    feat["n_train_eq_2"] = int(len(train) == 2)
    feat["n_train_ge_3"] = int(len(train) >= 3)
    feat["high_ratio"] = int(max(ratios) >= HIGH_RATIO_THRESHOLD)
    feat["large_test_input"] = int(max(x["cells"] for x in te_in_stats) >= LARGE_TEST_INPUT_CELLS)
    feat["many_colors"] = int(len(all_visible_colors) >= MANY_COLORS_THRESHOLD)
    feat["low_bg_train_input"] = int(
        np.mean([x["bg0_density"] for x in tr_in_stats]) < LOW_BG_THRESHOLD
    )
    return feat

def build_feature_df(data, split):
    return pd.DataFrame(
        [task_features(k, v, split) for k, v in data.items()]
    ).sort_values("task_id").reset_index(drop=True)

train_feat = build_feature_df(train_challenges, "train")
eval_feat  = build_feature_df(eval_challenges, "eval")
test_feat  = build_feature_df(test_challenges, "test")

for name, df in [("train", train_feat), ("eval", eval_feat), ("test", test_feat)]:
    df.to_csv(WORK_ROOT / f"task_features_{name}.csv", index=False)

numeric_cols = [
    c for c in eval_feat.columns
    if c not in {"task_id", "split"} and pd.api.types.is_numeric_dtype(eval_feat[c])
]
EXCLUDE_PROPENSITY = {
    "task_id", "split",
    "n_train_eq_2", "n_train_ge_3", "high_ratio", "large_test_input",
    "many_colors", "low_bg_train_input", "multi_query",
}
prop_cols = [c for c in numeric_cols if c not in EXCLUDE_PROPENSITY]

combined = pd.concat(
    [eval_feat.assign(_is_test=0), test_feat.assign(_is_test=1)],
    ignore_index=True
)
X = combined[prop_cols].astype(float)
y = combined["_is_test"].astype(int).to_numpy()

prop_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(
        C=0.5, penalty="l2", solver="liblinear",
        max_iter=5000, random_state=SEED
    )),
])
cv = StratifiedKFold(n_splits=N_PROPENSITY_FOLDS, shuffle=True, random_state=SEED)
p_oof = cross_val_predict(prop_model, X, y, cv=cv, method="predict_proba")[:, 1]

prop_auc = roc_auc_score(y, p_oof)
prop_logloss = log_loss(y, p_oof)

n_eval = len(eval_feat)
n_test = len(test_feat)
prior_eval = n_eval / (n_eval + n_test)
prior_test = n_test / (n_eval + n_test)

p_eval_as_test = np.clip(p_oof[:n_eval], 1e-4, 1 - 1e-4)
density_raw = (p_eval_as_test / (1 - p_eval_as_test)) * (prior_eval / prior_test)
density_clip = np.clip(density_raw, WEIGHT_CLIP_LOW, WEIGHT_CLIP_HIGH)
density_norm = density_clip / np.mean(density_clip)

eval_weights = eval_feat[["task_id"]].copy()
eval_weights["p_test_oof"] = p_eval_as_test
eval_weights["density_ratio_raw"] = density_raw
eval_weights["weight"] = density_norm
eval_weights["test_relevance_rank_pct"] = pd.Series(density_norm).rank(pct=True).to_numpy()
eval_weights["test_relevance_quartile"] = pd.qcut(
    pd.Series(density_norm).rank(method="first"),
    q=4,
    labels=["Q1_low", "Q2", "Q3", "Q4_high"]
).astype(str).to_numpy()

weight_ess = float((density_norm.sum() ** 2) / np.sum(density_norm ** 2))
eval_weights.to_csv(WORK_ROOT / "eval_test_propensity_weights.csv", index=False)

print("EXP-0 reproduction:")
print("  Eval-vs-Test CV AUC :", round(prop_auc, 6))
print("  Log loss            :", round(prop_logloss, 6))
print("  Weight ESS          :", round(weight_ess, 3), "/", len(eval_weights))
print("  Weight min/median/max:",
      round(float(density_norm.min()), 4),
      round(float(np.median(density_norm)), 4),
      round(float(density_norm.max()), 4))


# =====================================================================================
# SOURCE-ALIGNED PROXY-48 FEATURES
# Exact feature/tag rules copied from the source EDA notebook for audit comparability.
# These DO NOT replace EXP-0 features or official scoring.
# =====================================================================================
def source_eda_n_cells(grid):
    return int(np.prod(np.shape(grid)))

def source_eda_proxy_features(task_id, data):
    puzzle = data[task_id]
    train = puzzle["train"]
    input_sizes = [source_eda_n_cells(ex["input"]) for ex in train]
    output_sizes = [source_eda_n_cells(ex["output"]) for ex in train]
    ratios = [max(i, o) / max(min(i, o), 1) for i, o in zip(input_sizes, output_sizes)]
    colors = set()
    pixels = []
    for ex in train:
        for field in ["input", "output"]:
            a = np.asarray(ex[field])
            colors.update(np.unique(a).tolist())
            pixels.extend(a.reshape(-1).tolist())

    avg_input_size = float(np.mean(input_sizes))
    avg_ratio = float(np.mean(ratios))
    max_ratio = float(max(ratios))
    n_colors = len(colors)
    n_train = len(train)
    n_queries = len(puzzle["test"])
    is_dynamic = any(np.shape(ex["input"]) != np.shape(ex["output"]) for ex in train)
    is_rare = bool(6 in colors or 9 in colors)
    bg_density = pixels.count(0) / max(len(pixels), 1)

    if is_dynamic and max_ratio > 30:
        tier = 3
    elif is_dynamic and max_ratio > 5:
        tier = 2
    elif avg_input_size > 400:
        tier = 2
    else:
        tier = 1

    fine = (
        avg_input_size / 900 * 0.30 +
        (avg_ratio - 1) / 10 * 0.25 +
        n_colors / 10 * 0.20 +
        (0.25 if is_dynamic else 0.0)
    )
    if n_train >= 6 and tier < 3:
        fine *= 0.7

    tags = [f"tier{tier}"]
    if tier == 1 and (not is_dynamic) and max_ratio == 1 and n_colors <= 5:
        tags.append("easy_static")
    if tier == 1 and fine <= 0.22:
        tags.append("easy")
    if tier == 1 and fine > 0.45:
        tags.append("hard_tier1")
    if is_dynamic:
        tags.append("dynamic")
    if is_rare:
        tags.append("rare")
    if n_queries > 1:
        tags.append("multi_query")
    if max_ratio > 5:
        tags.append("high_ratio")
    if max_ratio > 30:
        tags.append("extreme_ratio")
    if n_colors >= 8:
        tags.append("many_colors")
    if avg_input_size >= 500:
        tags.append("large")
    if bg_density >= 0.70:
        tags.append("sparse_bg")
    if bg_density <= 0.30:
        tags.append("dense_non_bg")
    if n_train <= 2:
        tags.append("low_train")

    return {
        "task_id": task_id,
        "proxy_tier": int(tier),
        "proxy_fine": float(fine),
        "proxy_avg_input_size": avg_input_size,
        "proxy_max_ratio": max_ratio,
        "proxy_avg_ratio": avg_ratio,
        "proxy_n_colors": int(n_colors),
        "proxy_dynamic": int(is_dynamic),
        "proxy_rare": int(is_rare),
        "proxy_n_train": int(n_train),
        "proxy_n_queries": int(n_queries),
        "proxy_bg_density": float(bg_density),
        "proxy_easy_static": int("easy_static" in tags),
        "proxy_hard_tier1": int("hard_tier1" in tags),
        "proxy_high_ratio": int("high_ratio" in tags),
        "proxy_extreme_ratio": int("extreme_ratio" in tags),
        "proxy_tags": "|".join(tags),
    }

proxy_source_feat = pd.DataFrame([
    source_eda_proxy_features(k, eval_challenges) for k in eval_challenges
]).sort_values("task_id").reset_index(drop=True)
proxy_source_feat.to_csv(WORK_ROOT / "source_eda_features_eval.csv", index=False)

proxy48_source = proxy_source_feat[proxy_source_feat["task_id"].isin(PROXY48_TASK_IDS)].copy()
proxy48_order = {k: i for i, k in enumerate(PROXY48_TASK_IDS)}
proxy48_source["_order"] = proxy48_source["task_id"].map(proxy48_order)
proxy48_source = proxy48_source.sort_values("_order").drop(columns="_order").reset_index(drop=True)
proxy48_source.to_csv(WORK_ROOT / "proxy48_source_eda.csv", index=False)

proxy_source_counts = {
    "total": int(len(proxy48_source)),
    "tier1": int((proxy48_source["proxy_tier"] == 1).sum()),
    "tier2": int((proxy48_source["proxy_tier"] == 2).sum()),
    "tier3": int((proxy48_source["proxy_tier"] == 3).sum()),
    "easy_static": int(proxy48_source["proxy_easy_static"].sum()),
    "rare": int(proxy48_source["proxy_rare"].sum()),
    "multi_query": int((proxy48_source["proxy_n_queries"] > 1).sum()),
    "dynamic": int(proxy48_source["proxy_dynamic"].sum()),
    "hard_tier1": int(proxy48_source["proxy_hard_tier1"].sum()),
    "high_ratio": int(proxy48_source["proxy_high_ratio"].sum()),
    "extreme_ratio": int(proxy48_source["proxy_extreme_ratio"].sum()),
}
if proxy_source_counts != PROXY48_SOURCE_COUNTS:
    raise RuntimeError(
        "Proxy-48 source-EDA profile drift detected. "
        f"expected={PROXY48_SOURCE_COUNTS}, actual={proxy_source_counts}"
    )
with open(WORK_ROOT / "proxy48_source_distribution.json", "w", encoding="utf-8") as f:
    json.dump(proxy_source_counts, f, indent=2)
print("Proxy-48 source-EDA contract: PASS", proxy_source_counts)


EXP-0 reproduction:
  Eval-vs-Test CV AUC : 0.888229
  Log loss            : 0.42451
  Weight ESS          : 32.57 / 120
  Weight min/median/max: 0.2285 0.2285 5.7127
Proxy-48 source-EDA contract: PASS {'total': 48, 'tier1': 32, 'tier2': 12, 'tier3': 4, 'easy_static': 16, 'rare': 30, 'multi_query': 8, 'dynamic': 8, 'hard_tier1': 2, 'high_ratio': 5, 'extreme_ratio': 4}


In [4]:

# =====================================================================================
# CELL 4 — TOKENIZER + MINIMAL PROMPTS FOR BOTH MODES
# =====================================================================================
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True,
    trust_remote_code=False,
)
print("Tokenizer:", tokenizer.__class__.__name__, "| vocab:", len(tokenizer))

SYSTEM_PROMPT = """You are solving an ARC-AGI-2 task.
Infer the transformation rule from all training input-output examples and apply it to every test input.
Return the final answer only as valid JSON with this schema:
{"test_outputs":[GRID_0,GRID_1,...]}
Each GRID must be a rectangular 2D integer array using colors 0 through 9.
Return exactly one grid for each test input, in the same order."""

def grid_to_text(grid):
    return json.dumps(grid, separators=(",", ":"), ensure_ascii=False)

def build_user_prompt(task):
    # Intentionally DO NOT include public task_id in the model prompt.
    parts = []
    for i, ex in enumerate(task["train"]):
        parts += [
            f"TRAIN_{i}_INPUT", grid_to_text(ex["input"]),
            f"TRAIN_{i}_OUTPUT", grid_to_text(ex["output"]),
        ]
    for i, ex in enumerate(task["test"]):
        parts += [f"TEST_{i}_INPUT", grid_to_text(ex["input"])]
    parts.append(f"Return exactly {len(task['test'])} grid(s) in test_outputs.")
    return "\n".join(parts)

def render_chat(messages, mode):
    if mode == "off":
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            ), "enable_thinking=False"
        except Exception as exc:
            raise RuntimeError(
                "The mounted Qwen chat template did not accept enable_thinking=False. "
                f"Refusing to silently run the wrong control arm. Error: {exc}"
            )

    if mode == "xhigh":
        # Prefer explicit reasoning_effort. If this repacked template predates that kwarg,
        # enable_thinking=True still maps to Qwen's default xhigh behavior.
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=True,
                reasoning_effort="xhigh",
            ), 'enable_thinking=True, reasoning_effort="xhigh"'
        except Exception as exc1:
            print("WARNING: explicit reasoning_effort=xhigh was not accepted:", repr(exc1))
            try:
                return tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                    enable_thinking=True,
                ), "enable_thinking=True (template default effort)"
            except Exception as exc2:
                raise RuntimeError(
                    "Unable to render Qwen thinking prompt. "
                    f"Explicit xhigh error={exc1!r}; fallback error={exc2!r}"
                )
    raise ValueError(mode)

def make_prompt(task, mode):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(task)},
    ]
    return render_chat(messages, mode)

# Build every Eval prompt before model load.
prompt_records = []
render_modes = {}
for task_id, task in eval_challenges.items():
    for mode in ["off", "xhigh"]:
        prompt, render_method = make_prompt(task, mode)
        render_modes[mode] = render_method
        prompt_sha256 = hashlib.sha256(prompt.encode("utf-8")).hexdigest()
        prompt_records.append({
            "task_id": task_id,
            "mode": mode,
            "prompt": prompt,
            "prompt_sha256": prompt_sha256,
            "prompt_tokens": len(tokenizer.encode(prompt, add_special_tokens=False)),
            "n_train": len(task["train"]),
            "n_test": len(task["test"]),
        })

prompt_df = pd.DataFrame([{k:v for k,v in r.items() if k != "prompt"} for r in prompt_records])
prompt_df.to_csv(WORK_ROOT / "prompt_stats.csv", index=False)

print("Render methods:", render_modes)
print("\nPrompt token statistics:")
display(prompt_df.groupby("mode")["prompt_tokens"].describe(percentiles=[.5,.9,.95,.99]).round(1))

# Verify the two arms genuinely render different prompts.
probe_id = sorted(eval_challenges)[0]
probe_task = eval_challenges[probe_id]
probe_off, _ = make_prompt(probe_task, "off")
probe_on, _  = make_prompt(probe_task, "xhigh")
if probe_off == probe_on:
    raise RuntimeError("Thinking OFF and XHIGH rendered identical prompts. Abort.")
print("Thinking prompt toggle check: PASS")

# Save template fingerprint for reproducibility.
chat_template = getattr(tokenizer, "chat_template", "") or ""
template_sha = hashlib.sha256(chat_template.encode("utf-8")).hexdigest()
with open(WORK_ROOT / "chat_template_fingerprint.txt", "w", encoding="utf-8") as f:
    f.write(f"sha256={template_sha}\n")
    f.write(f"off_render_method={render_modes.get('off')}\n")
    f.write(f"xhigh_render_method={render_modes.get('xhigh')}\n")
print("Chat-template SHA256:", template_sha)


# =====================================================================================
# FROZEN ACTIVE TASK SELECTION
# PROXY48/PREFLIGHT use the exact source suite. SMOKE_GPU uses a deterministic subset OF it.
# =====================================================================================
def choose_smoke16_from_proxy():
    off_stats = (
        prompt_df[prompt_df["mode"] == "off"][["task_id", "prompt_tokens"]]
        .drop_duplicates("task_id")
        .set_index("task_id")
    )
    f = proxy_source_feat.set_index("task_id").join(off_stats, how="left")
    f = f.loc[[k for k in PROXY48_TASK_IDS if k in f.index]].copy()
    chosen = []

    def add(task_id):
        if task_id in f.index and task_id not in chosen:
            chosen.append(task_id)

    ordered = f.sort_values("prompt_tokens")
    positions = [0, int(.10*(len(ordered)-1)), int(.25*(len(ordered)-1)),
                 int(.50*(len(ordered)-1)), int(.75*(len(ordered)-1)),
                 int(.90*(len(ordered)-1)), len(ordered)-1]
    for p in positions:
        add(ordered.index[p])

    source_conditions = [
        lambda d: d["proxy_tier"] == 3,
        lambda d: d["proxy_dynamic"] == 1,
        lambda d: d["proxy_easy_static"] == 1,
        lambda d: d["proxy_rare"] == 1,
        lambda d: d["proxy_n_queries"] > 1,
        lambda d: d["proxy_high_ratio"] == 1,
        lambda d: d["proxy_hard_tier1"] == 1,
    ]
    med = float(f["prompt_tokens"].median())
    for cond in source_conditions:
        subset = f[cond(f)].copy()
        if len(subset):
            subset["_dist"] = (subset["prompt_tokens"] - med).abs()
            for task_id in subset.sort_values(["_dist", "prompt_tokens"]).index:
                if task_id not in chosen:
                    add(task_id)
                    break

    if len(chosen) < 16:
        remaining = [x for x in ordered.index if x not in chosen]
        idxs = np.linspace(0, len(remaining)-1, num=min(16-len(chosen), len(remaining)), dtype=int)
        for j in idxs:
            add(remaining[j])
    return chosen[:16]

if TASK_SET == "PROXY48":
    ACTIVE_TASK_IDS = list(PROXY48_TASK_IDS)
elif TASK_SET == "SMOKE16":
    ACTIVE_TASK_IDS = choose_smoke16_from_proxy()
else:
    raise RuntimeError(f"Unknown TASK_SET={TASK_SET}")

ACTIVE_TASK_SET = set(ACTIVE_TASK_IDS)
if len(ACTIVE_TASK_IDS) != ACTIVE_TASK_LIMIT or len(ACTIVE_TASK_SET) != ACTIVE_TASK_LIMIT:
    raise RuntimeError("Active task selection count/uniqueness mismatch.")
if not ACTIVE_TASK_SET.issubset(set(PROXY48_TASK_IDS)):
    raise RuntimeError("Q1 active set escaped the frozen Proxy-48 contract.")
if TASK_SET == "PROXY48" and ACTIVE_TASK_IDS != PROXY48_TASK_IDS:
    raise RuntimeError("Proxy-48 order drift detected.")

profile_task_table = eval_feat[eval_feat["task_id"].isin(ACTIVE_TASK_IDS)].copy()
profile_task_table = profile_task_table.merge(proxy_source_feat, on="task_id", how="left")
profile_task_table["prompt_tokens_off"] = profile_task_table["task_id"].map(
    prompt_df[prompt_df["mode"]=="off"].drop_duplicates("task_id").set_index("task_id")["prompt_tokens"]
)
profile_task_table["profile"] = RUN_PROFILE
profile_task_table["task_set"] = TASK_SET
profile_task_table["proxy_order"] = profile_task_table["task_id"].map(proxy48_order)
profile_task_table = profile_task_table.sort_values("proxy_order").reset_index(drop=True)
profile_task_table.to_csv(WORK_ROOT / "active_task_selection.csv", index=False)

proxy_contract = {
    "experiment_id": EXPERIMENT_ID,
    "generation_contract_version": GENERATION_CONTRACT_VERSION,
    "parser_contract_version": PARSER_CONTRACT_VERSION,
    "prompt_contract_version": PROMPT_CONTRACT_VERSION,
    "ordered_sha256": PROXY48_ORDERED_SHA256,
    "task_ids": PROXY48_TASK_IDS,
    "source_eda_counts": proxy_source_counts,
    "source_file_sha256": SOURCE_FILE_SHA256,
    "nvarc_diagnostic_anchors": NVARC_DIAGNOSTIC_ANCHORS,
}
with open(WORK_ROOT / "SOURCE_ALIGNMENT.json", "w", encoding="utf-8") as f:
    json.dump(proxy_contract, f, indent=2)

print("\nACTIVE TASKS:", len(ACTIVE_TASK_IDS), "|", TASK_SET)
print(ACTIVE_TASK_IDS)
print("Selected prompt token min/median/max:",
      int(profile_task_table["prompt_tokens_off"].min()),
      float(profile_task_table["prompt_tokens_off"].median()),
      int(profile_task_table["prompt_tokens_off"].max()))


Tokenizer: Qwen2TokenizerFast | vocab: 248077
Render methods: {'off': 'enable_thinking=False', 'xhigh': 'enable_thinking=True, reasoning_effort="xhigh"'}

Prompt token statistics:


,count,mean,std,min,50%,90%,95%,99%,max
mode,,,,,,,,,
off,120.0,5387.0,2807.5,1413.0,5036.0,8903.8,10264.0,15423.8,16385.0
xhigh,120.0,5423.0,2807.5,1449.0,5072.0,8939.8,10300.0,15459.8,16421.0


Thinking prompt toggle check: PASS
Chat-template SHA256: c3cf9e34abf4f9e36c2d72165aa9c132d3e2a725b6c2586aaa3a8af9d7a81041

ACTIVE TASKS: 48 | PROXY48
['0934a4d8', '20a9e565', '4c7dc4dd', '89565ca0', '136b0064', '291dc1e1', '16b78196', '271d71e2', '28a6681f', '2b83f449', '2c181942', '31f7f899', '35ab12c3', '3dc255db', '409aa875', '4c416de3', '64efde09', '6e453dd6', '7491f3cf', '7b0280bc', '7b80bb43', '7c66cb00', '1818057f', '332f06d7', '53fb4810', '7666fa5d', 'da515329', '8f3a5a89', 'db695cfb', '97d7923e', 'de809cff', 'c7f57c3e', 'cb2d8a2c', '58490d8a', 'dbff022c', '5961cc34', '80a900e0', '269e22fb', '8b9c3697', 'e376de54', 'b99e7126', 'c4d067a0', '135a2760', '4a21e3da', '71e489b6', '800d221b', 'dd6b8c4b', '36a08778']
Selected prompt token min/median/max: 1571 4781.5 11945


In [5]:
# =====================================================================================
# CELL 5 — CONTEXT / GENERATION SAFETY
# =====================================================================================
off_prompts = prompt_df[prompt_df["mode"]=="off"]["prompt_tokens"].to_numpy()
on_prompts  = prompt_df[prompt_df["mode"]=="xhigh"]["prompt_tokens"].to_numpy()

max_off_prompt = int(off_prompts.max())
max_on_prompt = int(on_prompts.max())

required_off = max_off_prompt + OFF_MAX_TOKENS + 512
required_xhigh = max_on_prompt + CONTEXT_PROBE_XHIGH_TOKENS + 512
desired_context = max(required_off, required_xhigh)
desired_context = int(math.ceil(desired_context / 1024) * 1024)

# Previously validated engine used 18,432. We expand only as much as direct XHIGH needs,
# capped conservatively at 32k for L4 KV-cache efficiency.
MAX_MODEL_LEN = min(32768, max(18432, desired_context))

safe_off_cap = MAX_MODEL_LEN - max_off_prompt - 512
safe_xhigh_cap = MAX_MODEL_LEN - max_on_prompt - 512

if safe_off_cap < 1024 or safe_xhigh_cap < 1024:
    raise RuntimeError(
        f"Insufficient context: off_safe={safe_off_cap}, xhigh_safe={safe_xhigh_cap}, "
        f"MAX_MODEL_LEN={MAX_MODEL_LEN}"
    )

EFFECTIVE_OFF_MAX_TOKENS = min(OFF_MAX_TOKENS, safe_off_cap)
EFFECTIVE_XHIGH_MAX_TOKENS = min(XHIGH_MAX_TOKENS, safe_xhigh_cap)

print("CONTEXT CONFIGURATION")
print("  max OFF prompt              :", max_off_prompt)
print("  max XHIGH prompt            :", max_on_prompt)
print("  desired context             :", desired_context)
print("  selected max_model_len      :", MAX_MODEL_LEN)
print("  OFF max generation          :", EFFECTIVE_OFF_MAX_TOKENS)
print("  XHIGH max generation        :", EFFECTIVE_XHIGH_MAX_TOKENS)
print("  max_num_batched_tokens      :", MAX_NUM_BATCHED_TOKENS)

if EFFECTIVE_OFF_MAX_TOKENS < OFF_MAX_TOKENS:
    print("WARNING: OFF generation ceiling reduced by context safety.")
if EFFECTIVE_XHIGH_MAX_TOKENS < XHIGH_MAX_TOKENS:
    print("WARNING: XHIGH generation ceiling reduced by context safety.")

context_config = {
    "max_off_prompt": max_off_prompt,
    "max_xhigh_prompt": max_on_prompt,
    "desired_context": desired_context,
    "max_model_len": MAX_MODEL_LEN,
    "off_requested_max_tokens": OFF_MAX_TOKENS,
    "off_effective_max_tokens": EFFECTIVE_OFF_MAX_TOKENS,
    "xhigh_requested_max_tokens": XHIGH_MAX_TOKENS,
    "context_probe_xhigh_tokens": CONTEXT_PROBE_XHIGH_TOKENS,
    "xhigh_effective_max_tokens": EFFECTIVE_XHIGH_MAX_TOKENS,
}
with open(WORK_ROOT / "context_config.json", "w", encoding="utf-8") as f:
    json.dump(context_config, f, indent=2)


CONTEXT CONFIGURATION
  max OFF prompt              : 16385
  max XHIGH prompt            : 16421
  desired context             : 29696
  selected max_model_len      : 29696
  OFF max generation          : 4096
  XHIGH max generation        : 12288
  max_num_batched_tokens      : 32768


In [6]:
# =====================================================================================
# CELL 6 — TOLERANT FINAL-GRID PARSER + SCORING UTILITIES + PRELOAD SELF-TESTS
# Interface rule: tolerate common envelopes, log exactly which parser path succeeded.
# =====================================================================================
import ast as py_ast
import re

def canonical_grid(grid):
    if not is_rectangular_grid(grid):
        return None
    return tuple(tuple(int(x) for x in row) for row in grid)

def canonical_grid_sha256(grid):
    cg = canonical_grid(grid)
    if cg is None:
        return None
    payload = json.dumps([list(r) for r in cg], separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def grids_equal(a, b):
    ca = canonical_grid(a)
    cb = canonical_grid(b)
    return ca is not None and cb is not None and ca == cb

def split_reasoning_and_answer(raw_text):
    text = raw_text or ""
    if "</think>" in text:
        reasoning, answer = text.rsplit("</think>", 1)
        reasoning = reasoning.replace("<think>", "", 1).strip()
        return reasoning, answer.strip(), "think_tags"
    return "", text.strip(), "no_visible_think_tags"

def strip_code_fence(text):
    t = text.strip()
    m = re.fullmatch(r"```(?:json|python|text)?\s*(.*?)\s*```", t, flags=re.I|re.S)
    return m.group(1).strip() if m else t

def iter_balanced_candidates(text, open_char, close_char):
    starts = [i for i, ch in enumerate(text) if ch == open_char]
    for start in starts:
        depth = 0
        in_str = False
        esc = False
        for i in range(start, len(text)):
            ch = text[i]
            if in_str:
                if esc:
                    esc = False
                elif ch == "\\":
                    esc = True
                elif ch == '"':
                    in_str = False
                continue
            if ch == '"':
                in_str = True
            elif ch == open_char:
                depth += 1
            elif ch == close_char:
                depth -= 1
                if depth == 0:
                    yield text[start:i+1]
                    break
                if depth < 0:
                    break

def normalize_test_outputs(obj, n_test):
    if isinstance(obj, dict) and "test_outputs" in obj:
        outs = obj["test_outputs"]
        schema = "dict_test_outputs"
    else:
        outs = obj
        schema = "direct_list"
    if not isinstance(outs, list):
        return None, schema
    if n_test == 1 and is_rectangular_grid(outs):
        return [outs], schema + "_single_grid"
    if len(outs) != n_test:
        return None, schema
    if not all(is_rectangular_grid(g) for g in outs):
        return None, schema
    return outs, schema

def parse_completion(raw_text, n_test, finish_reason=None):
    """
    Tolerant final-output parser with a truncation guard.

    Important Q1-v1.1 rule:
    - If generation ended by LENGTH, do NOT scan arbitrary reasoning prose for balanced
      arrays/objects. Those may be TRAIN/TEST grids quoted during reasoning and are not
      authoritative final answers.
    - On a length-truncated completion, accept only:
        (a) the entire completion if it is itself a valid payload, or
        (b) a payload explicitly following the last FINAL/ANSWER marker.
    - For normally stopped completions, retain the broader tolerant balanced-tail fallback.
    """
    reasoning, answer_text, reasoning_mode = split_reasoning_and_answer(raw_text)
    answer_text = strip_code_fence(answer_text)
    attempts = []
    hit_length_limit = str(finish_reason).lower() == "length"

    def add_json(name, text, confidence):
        try:
            attempts.append((name, json.loads(text), confidence))
        except Exception:
            pass

    def add_literal(name, text, confidence):
        try:
            attempts.append((name, py_ast.literal_eval(text), confidence))
        except Exception:
            pass

    # Highest confidence: whole completion is the requested payload.
    add_json("strict_json", answer_text, "high")
    add_literal("whole_python_literal", answer_text, "medium")

    # Prefer the tail after an explicit final-answer marker.
    marker_match = None
    for pattern in [
        r"(?is)final\s+answer\s*[:=-]",
        r"(?is)final\s*[:=-]",
        r"(?is)answer\s*[:=-]",
    ]:
        ms = list(re.finditer(pattern, answer_text))
        if ms:
            marker_match = ms[-1]
            break

    if marker_match:
        tail = strip_code_fence(answer_text[marker_match.end():].strip())
        add_json("final_marker_json", tail, "high")
        add_literal("final_marker_literal", tail, "medium")

        # A fenced/annotated marker tail may contain a little extra prose.
        obj_cands = list(iter_balanced_candidates(tail, "{", "}"))
        arr_cands = list(iter_balanced_candidates(tail, "[", "]"))
        for cand in reversed(obj_cands):
            add_json("final_marker_balanced_object", cand, "high")
            add_literal("final_marker_balanced_literal_object", cand, "medium")
        for cand in reversed(arr_cands):
            add_json("final_marker_balanced_array", cand, "medium")
            add_literal("final_marker_balanced_literal_array", cand, "medium")

    # General balanced scanning is useful for normally completed responses, but UNSAFE
    # after truncation because ARC reasoning often quotes many complete grids.
    if not hit_length_limit:
        obj_cands = list(iter_balanced_candidates(answer_text, "{", "}"))
        arr_cands = list(iter_balanced_candidates(answer_text, "[", "]"))
        for cand in reversed(obj_cands):
            add_json("balanced_json_object", cand, "medium")
            add_literal("balanced_literal_object", cand, "low")
        for cand in reversed(arr_cands):
            add_json("balanced_json_array", cand, "low")
            add_literal("balanced_literal_array", cand, "low")

    seen = set()
    for parser_name, obj, confidence in attempts:
        try:
            key = json.dumps(obj, sort_keys=True, default=str)
        except Exception:
            key = repr(obj)
        if key in seen:
            continue
        seen.add(key)
        outs, schema = normalize_test_outputs(obj, n_test)
        if outs is not None:
            return {
                "parse_valid": True,
                "parser": parser_name,
                "parser_confidence": confidence,
                "schema": schema,
                "test_outputs": outs,
                "reasoning_text": reasoning,
                "answer_text": answer_text,
                "reasoning_extraction": reasoning_mode,
                "truncation_guard_applied": hit_length_limit,
            }

    return {
        "parse_valid": False,
        "parser": "failed",
        "parser_confidence": "none",
        "schema": "",
        "test_outputs": [None] * n_test,
        "reasoning_text": reasoning,
        "answer_text": answer_text,
        "reasoning_extraction": reasoning_mode,
        "truncation_guard_applied": hit_length_limit,
    }

def token_count(text):
    if not text:
        return 0
    return len(tokenizer.encode(text, add_special_tokens=False))

def weighted_mean(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    return float(np.sum(values * weights) / np.sum(weights))

# =====================================================================================
# PRELOAD SELF-TESTS — BEFORE ANY 27B ALLOCATION
# =====================================================================================
def run_preload_self_tests():
    checks = {}
    p1 = parse_completion('{"test_outputs":[[[0,1],[1,0]]]}', 1)
    assert p1["parse_valid"] and p1["test_outputs"][0] == [[0,1],[1,0]]
    checks["parser_plain_json"] = True

    p2 = parse_completion(
        '<think>synthetic reasoning</think>\n{"test_outputs":[[[2,2],[2,2]]]}', 1
    )
    assert p2["parse_valid"] and "synthetic reasoning" in p2["reasoning_text"]
    checks["parser_thinking_json"] = True

    p3 = parse_completion('{"test_outputs":[[[1]],[[2,2],[2,2]]]}', 2)
    assert p3["parse_valid"] and len(p3["test_outputs"]) == 2
    checks["parser_multi_query"] = True

    p4 = parse_completion('{"test_outputs":[[[11]]]}', 1)
    assert not p4["parse_valid"]
    checks["parser_invalid_rejected"] = True

    p5 = parse_completion('FINAL ANSWER:\n```json\n[[[3,3],[3,3]]]\n```', 1)
    assert p5["parse_valid"] and p5["test_outputs"][0] == [[3,3],[3,3]]
    checks["parser_final_marker_fenced_grid"] = True

    p6 = parse_completion("{'test_outputs': [[[4]]]}", 1)
    assert p6["parse_valid"] and p6["test_outputs"][0] == [[4]]
    checks["parser_python_literal"] = True

    # v1.1 regression: never treat an intermediate reasoning grid as the final answer
    # when the model was cut off by max_tokens.
    p7 = parse_completion(
        "Reasoning... TRAIN grid [[1,2],[3,4]] ... still thinking ...",
        1,
        finish_reason="length",
    )
    assert not p7["parse_valid"]
    checks["parser_truncated_reasoning_rejected"] = True

    # But an explicit final marker remains authoritative even if the request technically
    # ends on the token boundary.
    p8 = parse_completion(
        "Reasoning... FINAL ANSWER: [[1,2],[3,4]]",
        1,
        finish_reason="length",
    )
    assert p8["parse_valid"] and p8["test_outputs"][0] == [[1,2],[3,4]]
    checks["parser_truncated_explicit_final_allowed"] = True

    assert np.isfinite(eval_weights["weight"]).all()
    assert abs(float(eval_weights["weight"].mean()) - 1.0) < 1e-8
    checks["propensity_weights"] = True

    assert PROXY48_ORDERED_SHA256 == PROXY48_EXPECTED_SHA256
    assert proxy_source_counts == PROXY48_SOURCE_COUNTS
    checks["proxy48_source_contract"] = True

    assert len(ACTIVE_TASK_IDS) == len(set(ACTIVE_TASK_IDS))
    assert all(t in eval_challenges and t in eval_solutions for t in ACTIVE_TASK_IDS)
    assert set(ACTIVE_TASK_IDS).issubset(set(PROXY48_TASK_IDS))
    checks["active_task_ids"] = True

    probe = WORK_ROOT / "_preload_write_probe.json"
    with open(probe, "w", encoding="utf-8") as f:
        json.dump({"profile": RUN_PROFILE, "ok": True}, f)
    with open(probe, "r", encoding="utf-8") as f:
        assert json.load(f)["ok"] is True
    probe.unlink()
    checks["filesystem"] = True

    with open(WORK_ROOT / "preload_selftest.json", "w", encoding="utf-8") as f:
        json.dump({"status": "PASS", "checks": checks}, f, indent=2)
    print("=" * 100)
    print("PRELOAD SELF-TEST: PASS")
    print(json.dumps(checks, indent=2))
    print("=" * 100)
    return checks

PRELOAD_CHECKS = run_preload_self_tests()


PRELOAD SELF-TEST: PASS
{
  "parser_plain_json": true,
  "parser_thinking_json": true,
  "parser_multi_query": true,
  "parser_invalid_rejected": true,
  "parser_final_marker_fenced_grid": true,
  "parser_python_literal": true,
  "parser_truncated_reasoning_rejected": true,
  "parser_truncated_explicit_final_allowed": true,
  "propensity_weights": true,
  "proxy48_source_contract": true,
  "active_task_ids": true,
  "filesystem": true
}


In [7]:
# =====================================================================================
# CELL 7 — CPU-SAFE PREFLIGHT OR REAL vLLM/TP=4 MODEL LOAD
# Dependency versions were already pinned in Cell 1 for PREFLIGHT/production parity.
# =====================================================================================
gpu_info = []
notebook_start = NOTEBOOK_WALL_START
model_load_sec = 0.0
warmup_sec = 0.0
arc_llm = None

RUNTIME_VLLM_VERSION = pkg_version("vllm")
RUNTIME_TORCH_VERSION = pkg_version("torch")
RUNTIME_TRANSFORMERS_VERSION = pkg_version("transformers")
RUNTIME_TOKENIZERS_VERSION = pkg_version("tokenizers")
RUNTIME_HF_HUB_VERSION = pkg_version("huggingface-hub")
RUNTIME_CUDA_VERSION = None

deadline = notebook_start + TOTAL_RUNTIME_SEC - SAFETY_RESERVE_SEC

# Fingerprint the mounted checkpoint without hashing the huge shard bytes.
with open(index_path, "r", encoding="utf-8") as f:
    _model_index = json.load(f)
_model_shards = sorted(set((_model_index.get("weight_map") or {}).values()))
_model_shard_layout = []
for _name in _model_shards:
    _p = MODEL_PATH / _name
    _model_shard_layout.append({
        "name": _name,
        "bytes": int(_p.stat().st_size) if _p.exists() else None,
    })

MODEL_ARTIFACT_FINGERPRINT_PAYLOAD = {
    "model_path": str(MODEL_PATH),
    "config_sha256": small_file_sha256(config_path),
    "safetensors_index_sha256": small_file_sha256(index_path),
    "shard_layout": _model_shard_layout,
    "chat_template_sha256": template_sha,
    "tokenizer_class": tokenizer.__class__.__name__,
    "tokenizer_vocab_size": len(tokenizer),
}
MODEL_ARTIFACT_FINGERPRINT_SHA256 = stable_json_sha256(MODEL_ARTIFACT_FINGERPRINT_PAYLOAD)

ENGINE_CONFIG = {
    "model": str(MODEL_PATH),
    "tensor_parallel_size": 4,
    "distributed_executor_backend": "mp",
    "dtype": "auto",
    "load_format": "safetensors",
    "trust_remote_code": False,
    "language_model_only": True,
    "max_model_len": MAX_MODEL_LEN,
    "max_num_seqs": MAX_NUM_SEQS,
    "max_num_batched_tokens": MAX_NUM_BATCHED_TOKENS,
    "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
    "enforce_eager": True,
    "disable_custom_all_reduce": True,
    "enable_prefix_caching": False,
    "seed": SEED,
}
ENGINE_CONFIG_SHA256 = stable_json_sha256(ENGINE_CONFIG)

if not USE_MODEL:
    # True no-allocation path: package parity is real, but do not import vLLM/CUDA runtime.
    SamplingParams = None
    print("=" * 116)
    print("PREFLIGHT CPU-SAFE / DEPENDENCY-PARITY MODE")
    print("=" * 116)
    print("vLLM package metadata :", RUNTIME_VLLM_VERSION)
    print("Transformers          :", RUNTIME_TRANSFORMERS_VERSION)
    print("Tokenizers            :", RUNTIME_TOKENIZERS_VERSION)
    print("HF Hub                :", RUNTIME_HF_HUB_VERSION)
    print("Skipping vLLM import, CUDA checks, SamplingParams API check and 27B allocation.")
    print("GPU/runtime-specific paths remain covered by SMOKE_GPU.")
else:
    import torch
    import vllm
    from vllm import LLM, SamplingParams

    RUNTIME_VLLM_VERSION = vllm.__version__
    RUNTIME_TORCH_VERSION = torch.__version__
    RUNTIME_CUDA_VERSION = torch.version.cuda

    print("=" * 116)
    print("REAL vLLM / CUDA PREFLIGHT")
    print("=" * 116)
    print("PyTorch :", RUNTIME_TORCH_VERSION)
    print("CUDA    :", RUNTIME_CUDA_VERSION)
    print("vLLM    :", RUNTIME_VLLM_VERSION)
    print("Transformers:", pkg_version("transformers"))

    if RUNTIME_VLLM_VERSION != EXPECTED_RUNTIME_PACKAGES["vllm"]:
        raise RuntimeError("vLLM runtime drift after import.")
    if pkg_version("transformers") != EXPECTED_RUNTIME_PACKAGES["transformers"]:
        raise RuntimeError("Transformers runtime drift after vLLM setup.")

    # Fail fast on the exact SamplingParams fields used later, BEFORE allocating 27B.
    _sampling_api_probe = SamplingParams(
        temperature=1.0,
        top_p=0.95,
        top_k=20,
        min_p=0.0,
        presence_penalty=0.0,
        repetition_penalty=1.0,
        max_tokens=16,
        ignore_eos=False,
        seed=SEED,
        skip_special_tokens=False,
        spaces_between_special_tokens=False,
    )
    print("SamplingParams API self-test: PASS")

    print("\nLOADING QWEN3.8-27B-FP8 ON 4×L4")
    print("GPUs visible:", torch.cuda.device_count())

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA unavailable. Select Kaggle 4×L4 accelerator.")
    if torch.cuda.device_count() != 4:
        raise RuntimeError(f"Expected 4 GPUs; PyTorch sees {torch.cuda.device_count()}.")

    for i in range(4):
        cc = torch.cuda.get_device_capability(i)
        mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
        name = torch.cuda.get_device_name(i)
        gpu_info.append({"gpu": i, "name": name, "memory_gib": mem, "cc": f"{cc[0]}.{cc[1]}"})
        print(f"GPU {i}: {name} | {mem:.2f} GiB | CC {cc[0]}.{cc[1]}")

    load_start = time.perf_counter()
    arc_llm = LLM(**ENGINE_CONFIG)
    model_load_sec = time.perf_counter() - load_start
    print(f"MODEL LOAD COMPLETE: {model_load_sec:.1f} sec")

    # Real generation smoke request catches engine/scheduler/model problems immediately.
    probe_record = min(
        [r for r in prompt_records if r["mode"] == "off" and r["task_id"] in ACTIVE_TASK_SET],
        key=lambda r: abs(r["prompt_tokens"] - np.median(off_prompts))
    )
    warm_params = SamplingParams(
        temperature=0.0,
        max_tokens=32,
        ignore_eos=False,
        skip_special_tokens=False,
    )
    t0 = time.perf_counter()
    warm = arc_llm.generate([probe_record["prompt"]], warm_params, use_tqdm=False)
    warmup_sec = time.perf_counter() - t0
    assert len(warm) == 1 and len(warm[0].outputs) >= 1
    print(f"REAL MODEL WARMUP: PASS ({warmup_sec:.2f} sec)")

RUNTIME_STACK = {
    "vllm": pkg_version("vllm"),
    "torch": pkg_version("torch"),
    "transformers": pkg_version("transformers"),
    "tokenizers": pkg_version("tokenizers"),
    "huggingface-hub": pkg_version("huggingface-hub"),
    "cuda": RUNTIME_CUDA_VERSION,
}
RUNTIME_STACK_SHA256 = stable_json_sha256(RUNTIME_STACK)

print("Model artifact fingerprint:", MODEL_ARTIFACT_FINGERPRINT_SHA256)
print("Engine config fingerprint :", ENGINE_CONFIG_SHA256)
print("Runtime stack fingerprint :", RUNTIME_STACK_SHA256)
print("Safety reserve:", SAFETY_RESERVE_SEC / 60, "minutes")


REAL vLLM / CUDA PREFLIGHT
PyTorch : 2.10.0+cu128
CUDA    : 12.8
vLLM    : 0.19.0
Transformers: 4.57.6
SamplingParams API self-test: PASS

LOADING QWEN3.8-27B-FP8 ON 4×L4
GPUs visible: 4
GPU 0: NVIDIA L4 | 22.03 GiB | CC 8.9
GPU 1: NVIDIA L4 | 22.03 GiB | CC 8.9
GPU 2: NVIDIA L4 | 22.03 GiB | CC 8.9
GPU 3: NVIDIA L4 | 22.03 GiB | CC 8.9
INFO 09-13 13:49:55 [utils.py:233] non-default args: {'load_format': 'safetensors', 'seed': 42, 'max_model_len': 29696, 'distributed_executor_backend': 'mp', 'tensor_parallel_size': 4, 'enable_prefix_caching': False, 'max_num_batched_tokens': 32768, 'max_num_seqs': 16, 'disable_log_stats': True, 'enforce_eager': True, 'disable_custom_all_reduce': True, 'language_model_only': True, 'model': '/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1'}
INFO 09-13 13:50:18 [model.py:549] Resolved architecture: Qwen3_5ForConditionalGeneration
INFO 09-13 13:50:18 [model.py:1678] Using max model len 29696
INFO 09-13 13:50:18 [scheduler

[W913 13:51:03.322232913 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W913 13:51:04.525881171 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W913 13:51:04.527286540 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W913 13:51:04.529717644 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
(Worker pid=585) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(Worker pid=584) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(Worker pid=586) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will b

(Worker pid=583) INFO 09-13 13:51:04 [pynccl.py:111] vLLM is using nccl==2.27.5
(Worker pid=583) WARNING 09-13 13:51:04 [symm_mem.py:66] SymmMemCommunicator: Device capability 8.9 not supported, communicator is not available.
(Worker pid=584) WARNING 09-13 13:51:04 [symm_mem.py:66] SymmMemCommunicator: Device capability 8.9 not supported, communicator is not available.
(Worker pid=585) WARNING 09-13 13:51:04 [symm_mem.py:66] SymmMemCommunicator: Device capability 8.9 not supported, communicator is not available.
(Worker pid=586) WARNING 09-13 13:51:04 [symm_mem.py:66] SymmMemCommunicator: Device capability 8.9 not supported, communicator is not available.
(Worker pid=583) INFO 09-13 13:51:04 [parallel_state.py:1716] rank 0 in world size 4 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A, EPLB rank N/A
(Worker_TP0 pid=583) INFO 09-13 13:51:05 [gpu_model_runner.py:4735] Starting to load model /kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch

Loading safetensors checkpoint shards:   0% Completed | 0/18 [00:00<?, ?it/s]


(Worker_TP0 pid=583) INFO 09-13 13:51:15 [weight_utils.py:825] Prefetching checkpoint files: 10% (1/5)


Loading safetensors checkpoint shards:   6% Completed | 1/18 [00:23<06:31, 23.03s/it]


... (9 outputs left out)

Loading safetensors checkpoint shards:  50% Completed | 9/18 [00:26<00:05,  1.58it/s]
Loading safetensors checkpoint shards:  56% Completed | 10/18 [00:26<00:03,  2.06it/s]


(Worker_TP2 pid=585) INFO 09-13 13:51:33 [weight_utils.py:825] Prefetching checkpoint files: 40% (4/4)
(Worker_TP2 pid=585) INFO 09-13 13:51:33 [weight_utils.py:843] Prefetching checkpoint files into page cache finished in 26.37s
(Worker_TP0 pid=583) INFO 09-13 13:51:33 [weight_utils.py:825] Prefetching checkpoint files: 50% (5/5)
(Worker_TP0 pid=583) INFO 09-13 13:51:33 [weight_utils.py:843] Prefetching checkpoint files into page cache finished in 26.46s


Loading safetensors checkpoint shards:  61% Completed | 11/18 [00:26<00:02,  2.60it/s]
Loading safetensors checkpoint shards:  67% Completed | 12/18 [00:26<00:01,  3.19it/s]
Loading safetensors checkpoint shards:  72% Completed | 13/18 [00:26<00:01,  3.78it/s]
Loading safetensors checkpoint shards:  78% Completed | 14/18 [00:27<00:00,  4.33it/s]
Loading safetensors checkpoint shards:  83% Completed | 15/18 [00:27<00:00,  4.82it/s]
Loading safetensors checkpoint shards:  89% Completed | 16/18 [00:27<00:00,  5.24it/s]
Loading safetensors checkpoint shards: 100% Completed | 18/18 [00:34<00:00,  1.79s/it]
Loading safetensors checkpoint shards: 100% Completed | 18/18 [00:34<00:00,  1.92s/it]
(Worker_TP0 pid=583) 


(Worker_TP0 pid=583) INFO 09-13 13:51:41 [default_loader.py:384] Loading weights took 34.68 seconds
(Worker_TP0 pid=583) INFO 09-13 13:51:42 [gpu_model_runner.py:4820] Model loading took 7.04 GiB memory and 35.296859 seconds
(Worker_TP1 pid=584) INFO 09-13 13:51:50 [weight_utils.py:825] Prefetching checkpoint files: 50% (5/5)
(Worker_TP1 pid=584) INFO 09-13 13:51:50 [weight_utils.py:843] Prefetching checkpoint files into page cache finished in 43.70s
(Worker_TP1 pid=584) WARNING 09-13 13:51:51 [fp8_utils.py:1185] Using default W8A8 Block FP8 kernel config. Performance might be sub-optimal! Config file not found at /usr/local/lib/python3.12/dist-packages/vllm/model_executor/layers/quantization/utils/configs/N=4096,K=5120,device_name=NVIDIA_L4,dtype=fp8_w8a8,block_shape=[128,128].json
(Worker_TP3 pid=586) WARNING 09-13 13:51:51 [fp8_utils.py:1185] Using default W8A8 Block FP8 kernel config. Performance might be sub-optimal! Config file not found at /usr/local/lib/python3.12/dist-packages

In [8]:
# =====================================================================================
# CELL 8 — GENERATION RUNNER WITH CONTRACT-SAFE CHECKPOINT / RESUME
# =====================================================================================
RAW_JSONL = WORK_ROOT / "raw_generations.jsonl"
CANDIDATE_EVENTS_JSONL = WORK_ROOT / "candidate_events.jsonl"

# Fixed, mode-appropriate parameters from the Qwen3.8 model-card recommendations.
MODE_CONFIG = {
    "xhigh": {
        "label": "thinking_xhigh",
        "batch_size": XHIGH_BATCH_SIZE,
        "max_tokens": int(EFFECTIVE_XHIGH_MAX_TOKENS),
        "temperature": 1.0,
        "top_p": 0.95,
        "top_k": 20,
        "min_p": 0.0,
        "presence_penalty": 0.0,
        "repetition_penalty": 1.0,
        "seed": SEED + 2000,
        "source": "Qwen3.8 official thinking recommendation",
    },
    "off": {
        "label": "thinking_off",
        "batch_size": OFF_BATCH_SIZE,
        "max_tokens": int(EFFECTIVE_OFF_MAX_TOKENS),
        "temperature": 0.7,
        "top_p": 0.80,
        "top_k": 20,
        "min_p": 0.0,
        "presence_penalty": 1.5,
        "repetition_penalty": 1.0,
        "seed": SEED + 1000,
        "source": "Qwen3.8 official non-thinking recommendation",
    },
}

QWEN_OFFICIAL_SAMPLING_REFERENCE = {
    "thinking": {
        "temperature": 1.0, "top_p": 0.95, "top_k": 20,
        "min_p": 0.0, "presence_penalty": 0.0, "repetition_penalty": 1.0,
    },
    "non_thinking": {
        "temperature": 0.7, "top_p": 0.80, "top_k": 20,
        "min_p": 0.0, "presence_penalty": 1.5, "repetition_penalty": 1.0,
    },
}

SAMPLING_FIELDS = [
    "batch_size", "max_tokens", "temperature", "top_p", "top_k",
    "min_p", "presence_penalty", "repetition_penalty", "seed",
]

PROMPT_RECORD_MAP = {
    (r["task_id"], r["mode"]): r
    for r in prompt_records
    if r["task_id"] in ACTIVE_TASK_SET
}

def sampling_config_payload(mode):
    cfg = MODE_CONFIG[mode]
    return {k: cfg[k] for k in SAMPLING_FIELDS}

SAMPLING_CONFIG_SHA256 = {
    mode: stable_json_sha256(sampling_config_payload(mode))
    for mode in MODE_CONFIG
}

def generation_contract_payload(prompt_rec, mode):
    return {
        "generation_contract_version": GENERATION_CONTRACT_VERSION,
        "parser_contract_version": PARSER_CONTRACT_VERSION,
        "prompt_contract_version": PROMPT_CONTRACT_VERSION,
        "experiment_id": EXPERIMENT_ID,
        "run_profile": RUN_PROFILE,
        "task_set": TASK_SET,
        "task_id": prompt_rec["task_id"],
        "mode": mode,
        "n_train": int(prompt_rec["n_train"]),
        "n_test": int(prompt_rec["n_test"]),
        "prompt_sha256": prompt_rec["prompt_sha256"],
        "prompt_tokens": int(prompt_rec["prompt_tokens"]),
        "render_method": render_modes[mode],
        "chat_template_sha256": template_sha,
        "model_artifact_fingerprint_sha256": MODEL_ARTIFACT_FINGERPRINT_SHA256,
        "runtime_stack_sha256": RUNTIME_STACK_SHA256,
        "engine_config_sha256": ENGINE_CONFIG_SHA256,
        "sampling_config": sampling_config_payload(mode),
    }

EXPECTED_CONTRACT_SHA256 = {
    key: stable_json_sha256(generation_contract_payload(rec, key[1]))
    for key, rec in PROMPT_RECORD_MAP.items()
}

def expected_contract_sha256(task_id, mode):
    return EXPECTED_CONTRACT_SHA256.get((task_id, mode))

def record_matches_expected(rec):
    """Strict resume gate. Old/stale/foreign rows never satisfy a current checkpoint."""
    task_id = rec.get("task_id")
    mode = rec.get("mode")
    key = (task_id, mode)
    if key not in EXPECTED_CONTRACT_SHA256:
        return False, "foreign_task_or_mode"

    prompt_rec = PROMPT_RECORD_MAP[key]
    cfg = MODE_CONFIG[mode]
    checks = [
        (rec.get("generation_contract_version") == GENERATION_CONTRACT_VERSION, "contract_version"),
        (rec.get("generation_contract_sha256") == EXPECTED_CONTRACT_SHA256[key], "contract_sha256"),
        (rec.get("prompt_sha256") == prompt_rec["prompt_sha256"], "prompt_sha256"),
        (rec.get("chat_template_sha256") == template_sha, "chat_template_sha256"),
        (rec.get("model_artifact_fingerprint_sha256") == MODEL_ARTIFACT_FINGERPRINT_SHA256, "model_fingerprint"),
        (rec.get("runtime_stack_sha256") == RUNTIME_STACK_SHA256, "runtime_stack"),
        (rec.get("engine_config_sha256") == ENGINE_CONFIG_SHA256, "engine_config"),
        (rec.get("sampling_config_sha256") == SAMPLING_CONFIG_SHA256[mode], "sampling_config"),
        (rec.get("seed") == cfg["seed"], "seed"),
        (rec.get("run_profile") == RUN_PROFILE, "run_profile"),
    ]
    for ok, reason in checks:
        if not ok:
            return False, reason
    return True, "compatible"

def read_existing_records():
    out = []
    malformed = 0
    if RAW_JSONL.exists():
        with open(RAW_JSONL, "r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, 1):
                if not line.strip():
                    continue
                try:
                    row = json.loads(line)
                    row["_checkpoint_line"] = line_no
                    out.append(row)
                except Exception:
                    malformed += 1
    if malformed:
        print(f"WARNING: ignored {malformed} malformed checkpoint JSONL line(s).")
    return out

def audit_checkpoint_records(records):
    compatible = []
    stale = []
    foreign = []
    for idx, rec in enumerate(records):
        ok, reason = record_matches_expected(rec)
        item = {
            "checkpoint_index": idx,
            "checkpoint_line": rec.get("_checkpoint_line"),
            "task_id": rec.get("task_id"),
            "mode": rec.get("mode"),
            "status": rec.get("status"),
            "run_id": rec.get("run_id"),
            "record_id": rec.get("record_id"),
            "reason": reason,
        }
        if ok:
            clean = dict(rec)
            clean["_checkpoint_order"] = idx
            compatible.append(clean)
        elif reason == "foreign_task_or_mode":
            foreign.append(item)
        else:
            stale.append(item)

    summary = {
        "total_rows": len(records),
        "compatible_rows": len(compatible),
        "stale_or_incompatible_rows": len(stale),
        "foreign_rows": len(foreign),
        "stale_preview": stale[:20],
        "foreign_preview": foreign[:20],
    }
    return compatible, summary

def compatible_success_keys(records):
    compatible, _ = audit_checkpoint_records(records)
    return {
        (r.get("task_id"), r.get("mode"))
        for r in compatible
        if r.get("status") == "ok"
    }

def append_jsonl(rows):
    with open(RAW_JSONL, "a", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")
        f.flush()
        os.fsync(f.fileno())

def append_candidate_events(generation_rows):
    """Persist per-test-output branch events immediately; use RAW provenance, never live config guesses."""
    events = []
    for rec in generation_rows:
        n_test = int(rec.get("n_test", 0) or 0)
        outs = rec.get("test_outputs") or []
        for qi in range(n_test):
            grid = outs[qi] if qi < len(outs) else None
            valid = canonical_grid(grid) is not None
            events.append({
                "run_id": rec.get("run_id"),
                "record_id": rec.get("record_id"),
                "task_id": rec.get("task_id"),
                "test_index": qi,
                "source": "q27_direct",
                "branch": rec.get("mode"),
                "seed": rec.get("seed"),
                "status": rec.get("status"),
                "grid": grid if valid else None,
                "grid_sha256": canonical_grid_sha256(grid),
                "valid_grid": bool(valid),
                "parse_valid": bool(rec.get("parse_valid", False)),
                "parser": rec.get("parser"),
                "parser_confidence": rec.get("parser_confidence"),
                "schema": rec.get("schema"),
                "prompt_tokens": rec.get("prompt_tokens"),
                "generated_tokens": rec.get("generated_tokens"),
                "wall_time_sec": rec.get("effective_batch_sec_per_task"),
                "prompt_sha256": rec.get("prompt_sha256"),
                "generation_contract_sha256": rec.get("generation_contract_sha256"),
                "sampling_config_sha256": rec.get("sampling_config_sha256"),
                "model_artifact_fingerprint_sha256": rec.get("model_artifact_fingerprint_sha256"),
                "raw_trace_path": "raw_generations.jsonl",
            })
    if not events:
        return
    with open(CANDIDATE_EVENTS_JSONL, "a", encoding="utf-8") as f:
        for event in events:
            f.write(json.dumps(event, ensure_ascii=False, default=str) + "\n")
        f.flush()
        os.fsync(f.fileno())

def write_progress(stage, mode=None, note=""):
    rows = read_existing_records()
    compatible, audit = audit_checkpoint_records(rows)
    ok = [r for r in compatible if r.get("status") == "ok"]
    payload = {
        "run_id": RUN_ID,
        "profile": RUN_PROFILE,
        "stage": stage,
        "mode": mode,
        "note": note,
        "compatible_ok_records": len(ok),
        "ok_xhigh": len({r.get("task_id") for r in ok if r.get("mode") == "xhigh"}),
        "ok_off": len({r.get("task_id") for r in ok if r.get("mode") == "off"}),
        "active_tasks": len(ACTIVE_TASK_IDS),
        "stale_rows_ignored": audit["stale_or_incompatible_rows"],
        "foreign_rows_ignored": audit["foreign_rows"],
        "updated_utc": datetime.now(timezone.utc).isoformat(),
    }
    with open(WORK_ROOT / "PROGRESS.json", "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

def output_runtime_metrics(request_output):
    result = {}
    metrics = getattr(request_output, "metrics", None)
    if metrics is None:
        return result
    for key in [
        "arrival_time", "first_scheduled_time", "first_token_time",
        "time_in_queue", "finished_time", "scheduler_time",
        "model_forward_time", "model_execute_time",
    ]:
        value = getattr(metrics, key, None)
        if value is not None:
            try:
                result[key] = float(value)
            except Exception:
                result[key] = str(value)
    return result

def make_sampling_params(mode):
    cfg = MODE_CONFIG[mode]
    kwargs = dict(
        temperature=cfg["temperature"],
        top_p=cfg["top_p"],
        top_k=cfg["top_k"],
        min_p=cfg["min_p"],
        presence_penalty=cfg["presence_penalty"],
        repetition_penalty=cfg["repetition_penalty"],
        max_tokens=cfg["max_tokens"],
        ignore_eos=False,
        seed=cfg["seed"],
        skip_special_tokens=False,
        spaces_between_special_tokens=False,
    )
    return SamplingParams(**kwargs)

def extract_request_output(req_out):
    outputs = getattr(req_out, "outputs", None) or []
    if not outputs:
        return {
            "status": "failed",
            "raw_text": "",
            "generated_tokens": 0,
            "finish_reason": "no_output",
            "stop_reason": None,
            "error": "vLLM request returned no candidate outputs",
        }
    o = outputs[0]
    return {
        "status": "ok",
        "raw_text": o.text or "",
        "generated_tokens": len(o.token_ids or []),
        "finish_reason": getattr(o, "finish_reason", None),
        "stop_reason": getattr(o, "stop_reason", None),
        "error": None,
    }

def base_generation_metadata(rec, mode, batch_id):
    cfg = MODE_CONFIG[mode]
    return {
        "experiment_id": EXPERIMENT_ID,
        "generation_contract_version": GENERATION_CONTRACT_VERSION,
        "parser_contract_version": PARSER_CONTRACT_VERSION,
        "prompt_contract_version": PROMPT_CONTRACT_VERSION,
        "run_profile": RUN_PROFILE,
        "run_id": RUN_ID,
        "task_id": rec["task_id"],
        "mode": mode,
        "mode_label": cfg["label"],
        "seed": cfg["seed"],
        "n_train": rec["n_train"],
        "n_test": rec["n_test"],
        "prompt_tokens": rec["prompt_tokens"],
        "prompt_sha256": rec["prompt_sha256"],
        "render_method": render_modes[mode],
        "chat_template_sha256": template_sha,
        "model_artifact_fingerprint_sha256": MODEL_ARTIFACT_FINGERPRINT_SHA256,
        "runtime_stack_sha256": RUNTIME_STACK_SHA256,
        "engine_config_sha256": ENGINE_CONFIG_SHA256,
        "sampling_config_sha256": SAMPLING_CONFIG_SHA256[mode],
        "generation_contract_sha256": EXPECTED_CONTRACT_SHA256[(rec["task_id"], mode)],
        "batch_id": batch_id,
    }

def run_checkpoint_contract_selftest():
    # 1) Exact current record is reusable.
    key = (ACTIVE_TASK_IDS[0], "off")
    rec = PROMPT_RECORD_MAP[key]
    good = base_generation_metadata(rec, "off", "selftest")
    good["status"] = "ok"
    ok, reason = record_matches_expected(good)
    assert ok and reason == "compatible"

    # 2) A prompt/config/notebook-contract mismatch is never reusable.
    stale = dict(good)
    stale["generation_contract_sha256"] = "0" * 64
    ok, _ = record_matches_expected(stale)
    assert not ok

    stale_seed = dict(good)
    stale_seed["seed"] = int(good["seed"]) + 1
    ok, _ = record_matches_expected(stale_seed)
    assert not ok

    stale_prompt = dict(good)
    stale_prompt["prompt_sha256"] = "f" * 64
    ok, _ = record_matches_expected(stale_prompt)
    assert not ok

    stale_model = dict(good)
    stale_model["model_artifact_fingerprint_sha256"] = "e" * 64
    ok, _ = record_matches_expected(stale_model)
    assert not ok

    stale_sampling = dict(good)
    stale_sampling["sampling_config_sha256"] = "d" * 64
    ok, _ = record_matches_expected(stale_sampling)
    assert not ok

    # 3) A foreign task cannot make current coverage complete.
    foreign = dict(good)
    foreign["task_id"] = "deadbeef"
    ok, reason = record_matches_expected(foreign)
    assert not ok and reason == "foreign_task_or_mode"

    # 4) Empty vLLM outputs are failed/incomplete, never status=ok.
    class _Dummy:
        outputs = []
    assert extract_request_output(_Dummy())["status"] == "failed"

    # 5) Missing active task + a foreign row cannot fake set-complete coverage.
    partial = []
    for task_id in ACTIVE_TASK_IDS[:-1]:
        p = PROMPT_RECORD_MAP[(task_id, "off")]
        r = base_generation_metadata(p, "off", "selftest")
        r["status"] = "ok"
        partial.append(r)
    partial.append(foreign)
    ok_tasks = {t for (t, m) in compatible_success_keys(partial) if m == "off"}
    assert ok_tasks != ACTIVE_TASK_SET
    assert ACTIVE_TASK_SET - ok_tasks == {ACTIVE_TASK_IDS[-1]}

    result = {
        "resume_exact_contract": True,
        "stale_contract_rejected": True,
        "stale_seed_rejected": True,
        "stale_prompt_rejected": True,
        "stale_model_rejected": True,
        "stale_sampling_rejected": True,
        "foreign_task_rejected": True,
        "no_output_failed": True,
        "exact_set_coverage_guard": True,
    }
    with open(WORK_ROOT / "checkpoint_contract_selftest.json", "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)
    print("Checkpoint/resume hardening self-test: PASS", result)

run_checkpoint_contract_selftest()

def run_mode(mode):
    cfg = MODE_CONFIG[mode]
    existing = read_existing_records()
    completed = compatible_success_keys(existing)

    records = [r for r in prompt_records if r["mode"] == mode and r["task_id"] in ACTIVE_TASK_SET]
    # Length bucketing reduces padding/head-of-line effects and makes reruns deterministic.
    records = sorted(records, key=lambda r: (r["prompt_tokens"], r["task_id"]))
    pending = [r for r in records if (r["task_id"], mode) not in completed]

    _, audit = audit_checkpoint_records(existing)
    print("\n" + "=" * 116)
    print(f"RUN MODE: {mode} | pending={len(pending)} / total={len(records)}")
    print("Config:", json.dumps(cfg, indent=2))
    print(
        "Checkpoint audit:",
        f"compatible={audit['compatible_rows']}, stale={audit['stale_or_incompatible_rows']}, "
        f"foreign={audit['foreign_rows']}"
    )
    print("=" * 116)

    if not pending:
        print("Nothing to do; every active task has a compatible successful checkpoint.")
        return

    params = make_sampling_params(mode)
    bs = cfg["batch_size"]

    for batch_idx in range(0, len(pending), bs):
        if time.perf_counter() >= deadline:
            print("GLOBAL DEADLINE REACHED — stopping generation cleanly.")
            return

        batch = pending[batch_idx:batch_idx+bs]
        prompts = [r["prompt"] for r in batch]
        batch_no = batch_idx // bs + 1
        batch_id = f"{RUN_ID}:{mode}:batch{batch_no:03d}"

        print(
            f"{mode} batch {batch_no:02d}/"
            f"{math.ceil(len(pending)/bs):02d} | n={len(batch)} | "
            f"prompt_tok min/med/max="
            f"{min(r['prompt_tokens'] for r in batch)}/"
            f"{np.median([r['prompt_tokens'] for r in batch]):.0f}/"
            f"{max(r['prompt_tokens'] for r in batch)}"
        )

        t0 = time.perf_counter()
        try:
            outputs = arc_llm.generate(prompts, params, use_tqdm=False)
            batch_wall = time.perf_counter() - t0

            if len(outputs) != len(batch):
                raise RuntimeError(
                    f"vLLM returned {len(outputs)} request outputs for a batch of {len(batch)} prompts."
                )

            batch_rows = []
            for rec, req_out in zip(batch, outputs):
                n_test = rec["n_test"]
                extracted = extract_request_output(req_out)
                raw_text = extracted["raw_text"]
                finish_reason = extracted["finish_reason"]

                parsed = parse_completion(raw_text, n_test, finish_reason=finish_reason)
                reasoning_tokens = token_count(parsed["reasoning_text"])
                answer_tokens = token_count(parsed["answer_text"])

                metadata = base_generation_metadata(rec, mode, batch_id)
                record_id = f"{RUN_ID}:{mode}:{rec['task_id']}:{batch_no:03d}"

                row = {
                    **metadata,
                    "record_id": record_id,
                    "recorded_utc": datetime.now(timezone.utc).isoformat(),
                    "status": extracted["status"],
                    "generated_tokens": extracted["generated_tokens"],
                    "reasoning_tokens_visible": reasoning_tokens,
                    "answer_tokens_visible": answer_tokens,
                    "finish_reason": finish_reason,
                    "stop_reason": extracted["stop_reason"],
                    "hit_length_limit": int(str(finish_reason).lower() == "length"),
                    "batch_wall_sec": batch_wall,
                    "batch_size_actual": len(batch),
                    "effective_batch_sec_per_task": batch_wall / len(batch),
                    "parse_valid": bool(parsed["parse_valid"]),
                    "parser": parsed["parser"],
                    "parser_confidence": parsed.get("parser_confidence"),
                    "schema": parsed["schema"],
                    "reasoning_extraction": parsed["reasoning_extraction"],
                    "truncation_guard_applied": parsed.get("truncation_guard_applied", False),
                    "test_outputs": parsed["test_outputs"],
                    "raw_completion": raw_text,
                    "reasoning_text": parsed["reasoning_text"],
                    "answer_text": parsed["answer_text"],
                    "rendered_prompt": rec["prompt"],
                    "request_metrics": output_runtime_metrics(req_out),
                }
                if extracted["error"]:
                    row["error"] = extracted["error"]
                batch_rows.append(row)

            append_jsonl(batch_rows)
            append_candidate_events(batch_rows)
            write_progress("batch_complete", mode=mode)
            print(
                f"  wall={batch_wall:.1f}s | ok={sum(r['status']=='ok' for r in batch_rows)}/{len(batch_rows)} | "
                f"parse-valid={sum(r['parse_valid'] for r in batch_rows)}/{len(batch_rows)} | "
                f"gen_tokens={sum(r['generated_tokens'] for r in batch_rows)}"
            )

        except Exception as exc:
            batch_wall = time.perf_counter() - t0
            err = f"{type(exc).__name__}: {exc}"
            print("  BATCH FAILED:", err)
            failure_rows = []
            for rec in batch:
                metadata = base_generation_metadata(rec, mode, batch_id)
                record_id = f"{RUN_ID}:{mode}:{rec['task_id']}:{batch_no:03d}:failed"
                failure_rows.append({
                    **metadata,
                    "record_id": record_id,
                    "recorded_utc": datetime.now(timezone.utc).isoformat(),
                    "status": "failed",
                    "generated_tokens": 0,
                    "reasoning_tokens_visible": 0,
                    "answer_tokens_visible": 0,
                    "finish_reason": "exception",
                    "stop_reason": None,
                    "hit_length_limit": 0,
                    "batch_wall_sec": batch_wall,
                    "batch_size_actual": len(batch),
                    "effective_batch_sec_per_task": batch_wall / len(batch),
                    "parse_valid": False,
                    "parser": "exception",
                    "parser_confidence": "none",
                    "schema": "",
                    "reasoning_extraction": "",
                    "truncation_guard_applied": False,
                    "test_outputs": [None] * rec["n_test"],
                    "raw_completion": "",
                    "reasoning_text": "",
                    "answer_text": "",
                    "rendered_prompt": rec["prompt"],
                    "request_metrics": {},
                    "error": err,
                    "traceback": traceback.format_exc(limit=3),
                })
            append_jsonl(failure_rows)
            append_candidate_events(failure_rows)
            write_progress("batch_failed", mode=mode, note=err)

def create_mock_preflight_records():
    """Exercise downstream pipeline without allocating the model.
    Uses TEST INPUT grids as dummy predictions; never uses target outputs to generate them.
    Accuracy from this profile is intentionally NON-INTERPRETABLE.
    """
    if RAW_JSONL.exists():
        RAW_JSONL.unlink()
    if CANDIDATE_EVENTS_JSONL.exists():
        CANDIDATE_EVENTS_JSONL.unlink()

    rows = []
    for mode in ["xhigh", "off"]:
        for rec in prompt_records:
            if rec["mode"] != mode or rec["task_id"] not in ACTIVE_TASK_SET:
                continue

            task = eval_challenges[rec["task_id"]]
            dummy_outputs = [ex["input"] for ex in task["test"]]
            answer = json.dumps({"test_outputs": dummy_outputs}, separators=(",", ":"))
            if mode == "xhigh":
                raw_text = "<think>synthetic PREFLIGHT reasoning trace</think>\n" + answer
            else:
                raw_text = answer

            parsed = parse_completion(raw_text, rec["n_test"], finish_reason="mock_preflight")
            assert parsed["parse_valid"]

            batch_id = f"{RUN_ID}:{mode}:mock"
            metadata = base_generation_metadata(rec, mode, batch_id)
            rows.append({
                **metadata,
                "record_id": f"{RUN_ID}:{mode}:{rec['task_id']}:mock",
                "recorded_utc": datetime.now(timezone.utc).isoformat(),
                "status": "ok",
                "generated_tokens": token_count(raw_text),
                "reasoning_tokens_visible": token_count(parsed["reasoning_text"]),
                "answer_tokens_visible": token_count(parsed["answer_text"]),
                "finish_reason": "mock_preflight",
                "stop_reason": None,
                "hit_length_limit": 0,
                "batch_wall_sec": 0.0,
                "batch_size_actual": 1,
                "effective_batch_sec_per_task": 0.0,
                "parse_valid": bool(parsed["parse_valid"]),
                "parser": parsed["parser"],
                "parser_confidence": parsed.get("parser_confidence"),
                "schema": parsed["schema"],
                "reasoning_extraction": parsed["reasoning_extraction"],
                "truncation_guard_applied": parsed.get("truncation_guard_applied", False),
                "test_outputs": parsed["test_outputs"],
                "raw_completion": raw_text,
                "reasoning_text": parsed["reasoning_text"],
                "answer_text": parsed["answer_text"],
                "rendered_prompt": rec["prompt"],
                "request_metrics": {},
                "mock_preflight": True,
            })

    append_jsonl(rows)
    append_candidate_events(rows)
    write_progress("mock_generation_complete", note="No 27B model was loaded.")
    print(f"Mock PREFLIGHT records written: {len(rows)}")

if USE_MODEL:
    # Primary reasoning arm first; OFF control second.
    run_mode("xhigh")
    run_mode("off")
else:
    create_mock_preflight_records()

print("\nGeneration stage complete.")
print("Checkpoint:", RAW_JSONL)


Checkpoint/resume hardening self-test: PASS {'resume_exact_contract': True, 'stale_contract_rejected': True, 'stale_seed_rejected': True, 'stale_prompt_rejected': True, 'stale_model_rejected': True, 'stale_sampling_rejected': True, 'foreign_task_rejected': True, 'no_output_failed': True, 'exact_set_coverage_guard': True}

RUN MODE: xhigh | pending=48 / total=48
Config: {
  "label": "thinking_xhigh",
  "batch_size": 8,
  "max_tokens": 12288,
  "temperature": 1.0,
  "top_p": 0.95,
  "top_k": 20,
  "min_p": 0.0,
  "presence_penalty": 0.0,
  "repetition_penalty": 1.0,
  "seed": 2042,
  "source": "Qwen3.8 official thinking recommendation"
}
Checkpoint audit: compatible=0, stale=0, foreign=0
xhigh batch 01/06 | n=8 | prompt_tok min/med/max=1607/2178/2557
  wall=1911.1s | ok=8/8 | parse-valid=0/8 | gen_tokens=98304
xhigh batch 02/06 | n=8 | prompt_tok min/med/max=2581/3321/3633
  wall=1908.7s | ok=8/8 | parse-valid=0/8 | gen_tokens=98304
xhigh batch 03/06 | n=8 | prompt_tok min/med/max=3791/4

In [9]:
# =====================================================================================
# CELL 9 — CONTRACT-FILTERED CHECKPOINT LOAD + EXACT COVERAGE + SCORING
# =====================================================================================
raw_records = read_existing_records()
raw_df = pd.DataFrame(raw_records)

print("Raw checkpoint rows:", len(raw_df))
if len(raw_df) == 0:
    raise RuntimeError("No generation records were produced.")

compatible_records, checkpoint_audit = audit_checkpoint_records(raw_records)
compatible_df = pd.DataFrame(compatible_records)

with open(WORK_ROOT / "checkpoint_audit.json", "w", encoding="utf-8") as f:
    json.dump(checkpoint_audit, f, indent=2, default=str)

print("Checkpoint audit:", json.dumps(checkpoint_audit, indent=2))

if len(compatible_df) == 0:
    raise RuntimeError(
        "Checkpoint contains no records compatible with the current prompt/model/config contract."
    )

# One authoritative record per expected (task, mode):
# prefer the latest compatible SUCCESS; if none exists, keep the latest compatible failed attempt.
compatible_df = compatible_df.reset_index(drop=True)
if "_checkpoint_order" not in compatible_df:
    compatible_df["_checkpoint_order"] = np.arange(len(compatible_df))

selected_rows = []
for (task_id, mode), g in compatible_df.groupby(["task_id", "mode"], sort=False):
    g = g.sort_values("_checkpoint_order")
    ok = g[g["status"] == "ok"]
    chosen = ok.iloc[-1] if len(ok) else g.iloc[-1]
    selected_rows.append(chosen.to_dict())
latest = pd.DataFrame(selected_rows)

# Keep only current experiment keys by construction.
lookup = {
    (row["task_id"], row["mode"]): row.to_dict()
    for _, row in latest.iterrows()
}

expected_task_count = len(ACTIVE_TASK_IDS)
coverage = {}
for mode in ["xhigh", "off"]:
    mode_compatible_ok = {
        r.get("task_id")
        for r in compatible_records
        if r.get("mode") == mode and r.get("status") == "ok"
    }
    raw_mode_ok = {
        r.get("task_id")
        for r in raw_records
        if r.get("mode") == mode and r.get("status") == "ok"
    }
    missing = sorted(ACTIVE_TASK_SET - mode_compatible_ok)
    extra_raw = sorted(t for t in (raw_mode_ok - ACTIVE_TASK_SET) if t is not None)
    stale_active = sorted({
        x.get("task_id")
        for x in checkpoint_audit["stale_preview"]
        if x.get("mode") == mode and x.get("task_id") in ACTIVE_TASK_SET
    })
    coverage[mode] = {
        "compatible_ok_tasks": len(mode_compatible_ok),
        "expected_tasks": expected_task_count,
        "complete": (mode_compatible_ok == ACTIVE_TASK_SET) and (len(extra_raw) == 0),
        "missing_tasks": missing,
        "extra_raw_ok_tasks": extra_raw,
        "stale_active_preview": stale_active,
    }

run_complete = all(v["complete"] for v in coverage.values())
print("Coverage:", json.dumps(coverage, indent=2))
print("RUN STATUS:", "COMPLETE" if run_complete else "PARTIAL")
print("ACCURACY INTERPRETABLE:", ACCURACY_IS_REAL)
if not run_complete:
    print("WARNING: scores below are PARTIAL because exact compatible coverage is incomplete.")

# Export current compatible evidence separately so stale rows in raw history cannot confuse downstream analysis.
with open(WORK_ROOT / "current_compatible_generations.jsonl", "w", encoding="utf-8") as f:
    for row in compatible_records:
        clean = {k: v for k, v in row.items() if not str(k).startswith("_checkpoint")}
        f.write(json.dumps(clean, ensure_ascii=False, default=str) + "\n")

with open(WORK_ROOT / "current_generation_records.jsonl", "w", encoding="utf-8") as f:
    for row in latest.to_dict(orient="records"):
        clean = {k: v for k, v in row.items() if not str(k).startswith("_checkpoint")}
        f.write(json.dumps(clean, ensure_ascii=False, default=str) + "\n")

weight_map = dict(zip(eval_weights["task_id"], eval_weights["weight"]))
quartile_map = dict(zip(eval_weights["task_id"], eval_weights["test_relevance_quartile"]))
feat_map = eval_feat.set_index("task_id").to_dict(orient="index")
proxy_feat_map = proxy_source_feat.set_index("task_id").to_dict(orient="index")

score_rows = []
for task_id in ACTIVE_TASK_IDS:
    targets = eval_solutions[task_id]
    off_rec = lookup.get((task_id, "off"))
    on_rec  = lookup.get((task_id, "xhigh"))

    off_outs = off_rec.get("test_outputs", []) if off_rec else []
    on_outs  = on_rec.get("test_outputs", []) if on_rec else []

    for qi, target in enumerate(targets):
        off_grid = off_outs[qi] if qi < len(off_outs) else None
        on_grid  = on_outs[qi] if qi < len(on_outs) else None

        off_exact = grids_equal(off_grid, target)
        on_exact = grids_equal(on_grid, target)
        both_valid = canonical_grid(off_grid) is not None and canonical_grid(on_grid) is not None
        attempts_equal = both_valid and canonical_grid(off_grid) == canonical_grid(on_grid)

        row = {
            "task_id": task_id,
            "test_index": qi,
            "weight": float(weight_map[task_id]),
            "test_relevance_quartile": quartile_map[task_id],
            "off_exact": int(off_exact),
            "xhigh_exact": int(on_exact),
            "union_pass2": int(off_exact or on_exact),
            "both_exact": int(off_exact and on_exact),
            "off_only": int(off_exact and not on_exact),
            "xhigh_only": int(on_exact and not off_exact),
            "neither": int(not off_exact and not on_exact),
            "attempts_equal": int(attempts_equal),
            "off_valid": int(canonical_grid(off_grid) is not None),
            "xhigh_valid": int(canonical_grid(on_grid) is not None),
            "off_grid": off_grid,
            "xhigh_grid": on_grid,
            "target": target,
        }
        for k, v in feat_map[task_id].items():
            if isinstance(v, (int, float, bool, np.integer, np.floating)):
                row[k] = v
        for k, v in proxy_feat_map[task_id].items():
            if k == "task_id":
                continue
            row[k] = v
        score_rows.append(row)

scores = pd.DataFrame(score_rows)

def summarize_mode(mode):
    exact_col = "off_exact" if mode == "off" else "xhigh_exact"
    mode_rows = latest[latest["mode"] == mode].copy()
    return {
        "mode": mode,
        "task_outputs_scored": len(scores),
        "raw_accuracy": float(scores[exact_col].mean()),
        "test_weighted_accuracy": weighted_mean(scores[exact_col], scores["weight"]),
        "valid_output_pct": float(100 * scores[
            "off_valid" if mode=="off" else "xhigh_valid"
        ].mean()),
        "task_generation_records": int(len(mode_rows)),
        "status_ok_task_pct": float(100 * (mode_rows["status"] == "ok").mean()) if len(mode_rows) else 0.0,
        "parse_valid_task_pct": float(100 * mode_rows["parse_valid"].fillna(False).mean()) if len(mode_rows) else 0.0,
        "length_limit_task_pct": float(100 * mode_rows["hit_length_limit"].fillna(0).astype(float).mean()) if len(mode_rows) else 0.0,
        "no_output_failures": int((mode_rows["finish_reason"] == "no_output").sum()) if len(mode_rows) else 0,
        "avg_prompt_tokens": float(mode_rows["prompt_tokens"].mean()) if len(mode_rows) else np.nan,
        "avg_generated_tokens": float(mode_rows["generated_tokens"].mean()) if len(mode_rows) else np.nan,
        "median_generated_tokens": float(mode_rows["generated_tokens"].median()) if len(mode_rows) else np.nan,
        "avg_visible_reasoning_tokens": float(mode_rows["reasoning_tokens_visible"].mean()) if len(mode_rows) else np.nan,
        "avg_visible_answer_tokens": float(mode_rows["answer_tokens_visible"].mean()) if len(mode_rows) else np.nan,
        "sum_generated_tokens": int(mode_rows["generated_tokens"].sum()) if len(mode_rows) else 0,
        "approx_generation_batch_wall_sec_sum": float(
            mode_rows.drop_duplicates(["batch_id"])["batch_wall_sec"].sum()
        ) if len(mode_rows) and "batch_id" in mode_rows else np.nan,
    }

mode_summary = pd.DataFrame([summarize_mode("off"), summarize_mode("xhigh")])

union_summary = {
    "mode": "union_off_xhigh",
    "task_outputs_scored": int(len(scores)),
    "raw_accuracy": float(scores["union_pass2"].mean()),
    "test_weighted_accuracy": weighted_mean(scores["union_pass2"], scores["weight"]),
    "attempt2_increment_over_off": float((scores["union_pass2"] - scores["off_exact"]).mean()),
    "attempt2_increment_over_xhigh": float((scores["union_pass2"] - scores["xhigh_exact"]).mean()),
    "attempts_equal_pct": float(100 * scores["attempts_equal"].mean()),
    "off_only_correct_outputs": int(scores["off_only"].sum()),
    "xhigh_only_correct_outputs": int(scores["xhigh_only"].sum()),
    "both_correct_outputs": int(scores["both_exact"].sum()),
    "neither_correct_outputs": int(scores["neither"].sum()),
}

print("\nMODE SUMMARY")
display(mode_summary.round(4))
print("\nUNION SUMMARY")
print(json.dumps(union_summary, indent=2))

scores.to_csv(WORK_ROOT / "task_output_scores.csv", index=False)
mode_summary.to_csv(WORK_ROOT / "mode_summary.csv", index=False)
with open(WORK_ROOT / "union_summary.json", "w", encoding="utf-8") as f:
    json.dump(union_summary, f, indent=2)


# -------------------------------------------------------------------------------------
# Canonical candidate registry + exact-grid clusters (generation evidence before judging)
# -------------------------------------------------------------------------------------
candidate_rows = []
for _, rec in latest.iterrows():
    mode = rec.get("mode")
    n_test = int(rec.get("n_test", 0) or 0)
    outs = rec.get("test_outputs") or []
    for qi in range(n_test):
        grid = outs[qi] if qi < len(outs) else None
        cg = canonical_grid(grid)
        candidate_rows.append({
            "task_id": rec.get("task_id"),
            "test_index": qi,
            "source": "q27_direct",
            "branch": mode,
            "run_id": rec.get("run_id"),
            "record_id": rec.get("record_id"),
            "seed": rec.get("seed"),
            "status": rec.get("status"),
            "grid": grid if cg is not None else None,
            "grid_sha256": canonical_grid_sha256(grid),
            "valid_grid": bool(cg is not None),
            "shape": list(np.asarray(grid).shape) if cg is not None else None,
            "parse_valid": bool(rec.get("parse_valid", False)),
            "parser": rec.get("parser"),
            "parser_confidence": rec.get("parser_confidence"),
            "schema": rec.get("schema"),
            "prompt_tokens": rec.get("prompt_tokens"),
            "generated_tokens": rec.get("generated_tokens"),
            "wall_time_sec": rec.get("effective_batch_sec_per_task"),
            "prompt_sha256": rec.get("prompt_sha256"),
            "generation_contract_sha256": rec.get("generation_contract_sha256"),
            "sampling_config_sha256": rec.get("sampling_config_sha256"),
            "model_artifact_fingerprint_sha256": rec.get("model_artifact_fingerprint_sha256"),
            "runtime_stack_sha256": rec.get("runtime_stack_sha256"),
            "engine_config_sha256": rec.get("engine_config_sha256"),
            "raw_trace_path": "raw_generations.jsonl",
            "hypothesis_summary": None,
            "program_text": None,
            "program_hash": None,
            "train_exact": None,
            "parent_ids": [],
        })

registry_df = pd.DataFrame(candidate_rows)
if len(registry_df):
    registry_df["_order"] = np.arange(len(registry_df))
    registry_df = registry_df.sort_values("_order").drop_duplicates(
        ["task_id", "test_index", "branch"], keep="last"
    ).drop(columns="_order").reset_index(drop=True)

with open(WORK_ROOT / "candidate_registry.jsonl", "w", encoding="utf-8") as f:
    for row in registry_df.to_dict(orient="records"):
        f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")

registry_csv = registry_df.copy()
for col in ["grid", "shape", "parent_ids"]:
    if col in registry_csv:
        registry_csv[col] = registry_csv[col].apply(
            lambda x: json.dumps(x, separators=(",", ":"), default=str)
        )
registry_csv.to_csv(WORK_ROOT / "candidate_registry.csv", index=False)

clusters = []
valid_registry = registry_df[registry_df["valid_grid"] == True].copy() if len(registry_df) else registry_df
if len(valid_registry):
    for (task_id, qi, grid_sha), g in valid_registry.groupby(
        ["task_id", "test_index", "grid_sha256"], dropna=False
    ):
        provenance_cols = [
            "source", "branch", "run_id", "record_id", "seed", "parser", "parser_confidence",
            "prompt_sha256", "generation_contract_sha256", "sampling_config_sha256",
            "model_artifact_fingerprint_sha256",
        ]
        clusters.append({
            "task_id": task_id,
            "test_index": int(qi),
            "grid_sha256": grid_sha,
            "grid": g.iloc[0]["grid"],
            "source_run_ids": sorted(set(str(x) for x in g["run_id"].dropna().tolist())),
            "provenance": g[provenance_cols].to_dict(orient="records"),
            "independent_branch_count": int(g["branch"].nunique()),
        })
with open(WORK_ROOT / "candidate_clusters.jsonl", "w", encoding="utf-8") as f:
    for row in clusters:
        f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")
print("Candidate registry rows:", len(registry_df), "| exact-grid clusters:", len(clusters))

# Strict task-level diagnostic: all test queries of a task must be correct.
task_scores = scores.groupby("task_id").agg(
    off_all_queries=("off_exact", "min"),
    xhigh_all_queries=("xhigh_exact", "min"),
    union_all_queries=("union_pass2", "min"),
    n_queries=("test_index", "count"),
).reset_index()
task_scores.to_csv(WORK_ROOT / "strict_task_level_scores.csv", index=False)

print("\nStrict all-query task accuracy:")
print("  OFF   :", round(task_scores["off_all_queries"].mean(), 4))
print("  XHIGH :", round(task_scores["xhigh_all_queries"].mean(), 4))
print("  UNION :", round(task_scores["union_all_queries"].mean(), 4))


Raw checkpoint rows: 96
Checkpoint audit: {
  "total_rows": 96,
  "compatible_rows": 96,
  "stale_or_incompatible_rows": 0,
  "foreign_rows": 0,
  "stale_preview": [],
  "foreign_preview": []
}
Coverage: {
  "xhigh": {
    "compatible_ok_tasks": 48,
    "expected_tasks": 48,
    "complete": true,
    "missing_tasks": [],
    "extra_raw_ok_tasks": [],
    "stale_active_preview": []
  },
  "off": {
    "compatible_ok_tasks": 48,
    "expected_tasks": 48,
    "complete": true,
    "missing_tasks": [],
    "extra_raw_ok_tasks": [],
    "stale_active_preview": []
  }
}
RUN STATUS: COMPLETE
ACCURACY INTERPRETABLE: True

MODE SUMMARY


,mode,task_outputs_scored,raw_accuracy,test_weighted_accuracy,valid_output_pct,task_generation_records,status_ok_task_pct,parse_valid_task_pct,length_limit_task_pct,no_output_failures,avg_prompt_tokens,avg_generated_tokens,median_generated_tokens,avg_visible_reasoning_tokens,avg_visible_answer_tokens,sum_generated_tokens,approx_generation_batch_wall_sec_sum
0,off,56,0.0,0.0,55.3571,48,100.0,60.4167,37.5,0,5050.4583,2217.4583,1502.0,0.0,2216.7292,106438,2068.9140
1,xhigh,56,0.0,0.0,0.0000,48,100.0,0.0000,100.0,0,5086.4583,12288.0000,12288.0,0.0,12287.9583,589824,11517.2621



UNION SUMMARY
{
  "mode": "union_off_xhigh",
  "task_outputs_scored": 56,
  "raw_accuracy": 0.0,
  "test_weighted_accuracy": 0.0,
  "attempt2_increment_over_off": 0.0,
  "attempt2_increment_over_xhigh": 0.0,
  "attempts_equal_pct": 0.0,
  "off_only_correct_outputs": 0,
  "xhigh_only_correct_outputs": 0,
  "both_correct_outputs": 0,
  "neither_correct_outputs": 56
}
Candidate registry rows: 112 | exact-grid clusters: 31

Strict all-query task accuracy:
  OFF   : 0.0
  XHIGH : 0.0
  UNION : 0.0


In [10]:
# =====================================================================================
# CELL 10 — SOURCE-ALIGNED PROXY LENSES + SECONDARY STRUCTURAL BREAKDOWNS
# =====================================================================================
CATEGORY_MASKS = {
    "ALL": lambda d: np.ones(len(d), dtype=bool),
    # Frozen EDA source lenses — primary subgroup reporting for Proxy-48.
    "proxy_tier1": lambda d: (d["proxy_tier"] == 1).to_numpy(),
    "proxy_tier2": lambda d: (d["proxy_tier"] == 2).to_numpy(),
    "proxy_tier3": lambda d: (d["proxy_tier"] == 3).to_numpy(),
    "proxy_easy_static": lambda d: d["proxy_easy_static"].astype(bool).to_numpy(),
    "proxy_rare": lambda d: d["proxy_rare"].astype(bool).to_numpy(),
    "proxy_multi_query": lambda d: (d["proxy_n_queries"] > 1).to_numpy(),
    "proxy_dynamic": lambda d: d["proxy_dynamic"].astype(bool).to_numpy(),
    "proxy_hard_tier1": lambda d: d["proxy_hard_tier1"].astype(bool).to_numpy(),
    "proxy_high_ratio_gt5": lambda d: d["proxy_high_ratio"].astype(bool).to_numpy(),
    "proxy_extreme_ratio_gt30": lambda d: d["proxy_extreme_ratio"].astype(bool).to_numpy(),
    # Existing EXP-1 diagnostics retained under explicit names.
    "exp1_n_train_eq_2": lambda d: d["n_train_eq_2"].astype(bool).to_numpy(),
    "exp1_same_shape_all_train": lambda d: (d["same_shape_frac"] == 1.0).to_numpy(),
    "exp1_high_area_ratio_ge_10": lambda d: d["high_ratio"].astype(bool).to_numpy(),
    "exp1_large_test_input_ge_300_cells": lambda d: d["large_test_input"].astype(bool).to_numpy(),
    "exp1_visible_colors_ge_7": lambda d: d["many_colors"].astype(bool).to_numpy(),
}

cat_rows = []
for name, fn in CATEGORY_MASKS.items():
    mask = fn(scores)
    s = scores.loc[mask]
    if len(s) == 0:
        continue
    cat_rows.append({
        "category": name,
        "task_outputs": len(s),
        "tasks": s["task_id"].nunique(),
        "off_raw": s["off_exact"].mean(),
        "xhigh_raw": s["xhigh_exact"].mean(),
        "union_raw": s["union_pass2"].mean(),
        "off_weighted": weighted_mean(s["off_exact"], s["weight"]),
        "xhigh_weighted": weighted_mean(s["xhigh_exact"], s["weight"]),
        "union_weighted": weighted_mean(s["union_pass2"], s["weight"]),
        "off_only": int(s["off_only"].sum()),
        "xhigh_only": int(s["xhigh_only"].sum()),
        "attempts_equal_pct": 100 * s["attempts_equal"].mean(),
    })

# EXP-0 propensity quartiles remain exploratory only.
for q, s in scores.groupby("test_relevance_quartile"):
    cat_rows.append({
        "category": f"propensity_{q}",
        "task_outputs": len(s),
        "tasks": s["task_id"].nunique(),
        "off_raw": s["off_exact"].mean(),
        "xhigh_raw": s["xhigh_exact"].mean(),
        "union_raw": s["union_pass2"].mean(),
        "off_weighted": weighted_mean(s["off_exact"], s["weight"]),
        "xhigh_weighted": weighted_mean(s["xhigh_exact"], s["weight"]),
        "union_weighted": weighted_mean(s["union_pass2"], s["weight"]),
        "off_only": int(s["off_only"].sum()),
        "xhigh_only": int(s["xhigh_only"].sum()),
        "attempts_equal_pct": 100 * s["attempts_equal"].mean(),
    })

category_scores = pd.DataFrame(cat_rows)
category_scores.to_csv(WORK_ROOT / "category_scores.csv", index=False)
print("CATEGORY BREAKDOWN")
display(category_scores.round(4))


CATEGORY BREAKDOWN


,category,task_outputs,tasks,off_raw,xhigh_raw,union_raw,off_weighted,xhigh_weighted,union_weighted,off_only,xhigh_only,attempts_equal_pct
0,ALL,56,48,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0
1,proxy_tier1,38,32,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0
2,proxy_tier2,12,12,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0
3,proxy_tier3,6,4,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0
4,proxy_easy_static,21,16,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0
5,proxy_rare,34,30,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0
6,proxy_multi_query,16,8,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0
7,proxy_dynamic,11,8,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0
8,proxy_hard_tier1,2,2,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0
9,proxy_high_ratio_gt5,7,5,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0


In [11]:

# =====================================================================================
# CELL 11 — DIVERSITY / OVERLAP / FAILURE TAXONOMY
# =====================================================================================
def key_col(row):
    return f"{row['task_id']}:{int(row['test_index'])}"

scores["output_key"] = scores.apply(key_col, axis=1)

overlap = {
    "both_correct": scores.loc[scores["both_exact"]==1, "output_key"].tolist(),
    "off_only": scores.loc[scores["off_only"]==1, "output_key"].tolist(),
    "xhigh_only": scores.loc[scores["xhigh_only"]==1, "output_key"].tolist(),
    "neither": scores.loc[scores["neither"]==1, "output_key"].tolist(),
    "different_valid_answers": scores.loc[
        (scores["off_valid"]==1) & (scores["xhigh_valid"]==1) & (scores["attempts_equal"]==0),
        "output_key"
    ].tolist(),
}
with open(WORK_ROOT / "correctness_overlap.json", "w", encoding="utf-8") as f:
    json.dump(overlap, f, indent=2)

failure_rows = []
for _, r in latest.iterrows():
    failure_rows.append({
        "task_id": r["task_id"],
        "mode": r["mode"],
        "status": r.get("status"),
        "parse_valid": r.get("parse_valid"),
        "parser": r.get("parser"),
        "parser_confidence": r.get("parser_confidence"),
        "finish_reason": r.get("finish_reason"),
        "hit_length_limit": r.get("hit_length_limit"),
        "prompt_tokens": r.get("prompt_tokens"),
        "generated_tokens": r.get("generated_tokens"),
        "reasoning_tokens_visible": r.get("reasoning_tokens_visible"),
        "answer_tokens_visible": r.get("answer_tokens_visible"),
        "error": r.get("error", ""),
    })
failure_df = pd.DataFrame(failure_rows)
failure_df.to_csv(WORK_ROOT / "generation_diagnostics.csv", index=False)

print("Overlap counts:")
for k, v in overlap.items():
    print(f"  {k:24s}: {len(v)}")

print("\nGeneration diagnostics by mode:")
display(
    failure_df.groupby("mode").agg(
        records=("task_id","count"),
        parse_valid_pct=("parse_valid", lambda x: 100*np.mean(pd.Series(x).fillna(False))),
        length_limit_pct=("hit_length_limit", lambda x: 100*np.mean(pd.to_numeric(x, errors="coerce").fillna(0))),
        avg_generated_tokens=("generated_tokens","mean"),
        avg_reasoning_tokens=("reasoning_tokens_visible","mean"),
    ).round(2)
)


Overlap counts:
  both_correct            : 0
  off_only                : 0
  xhigh_only              : 0
  neither                 : 56
  different_valid_answers : 0

Generation diagnostics by mode:


,records,parse_valid_pct,length_limit_pct,avg_generated_tokens,avg_reasoning_tokens
mode,,,,,
off,48,60.42,37.5,2217.46,0.0
xhigh,48,0.00,100.0,12288.00,0.0


In [12]:

# =====================================================================================
# CELL 12 — WRITE OFFICIAL-STYLE PREDICTION FILES FOR REPLAY / EXP-0 SCORER
# =====================================================================================
def outputs_for(task_id, mode):
    rec = lookup.get((task_id, mode))
    if not rec:
        return [None] * len(eval_challenges[task_id]["test"])
    outs = rec.get("test_outputs", [])
    return [
        outs[i] if i < len(outs) and is_rectangular_grid(outs[i]) else None
        for i in range(len(eval_challenges[task_id]["test"]))
    ]

def fallback_grid(task_id, qi):
    # Only for schema-valid replay files. Never count this as a model candidate.
    inp = eval_challenges[task_id]["test"][qi]["input"]
    return [[0 for _ in row] for row in inp]

def schema_grid(candidate, task_id, qi):
    return candidate if is_rectangular_grid(candidate) else fallback_grid(task_id, qi)

off_acc = float(scores["off_exact"].mean())
on_acc = float(scores["xhigh_exact"].mean())
best_first = "xhigh" if on_acc >= off_acc else "off"
other = "off" if best_first == "xhigh" else "xhigh"

pred_off = {}
pred_xhigh = {}
pred_union = {}

for task_id in ACTIVE_TASK_IDS:
    task = eval_challenges[task_id]
    off_out = outputs_for(task_id, "off")
    on_out = outputs_for(task_id, "xhigh")
    pred_off[task_id] = []
    pred_xhigh[task_id] = []
    pred_union[task_id] = []

    for qi in range(len(task["test"])):
        og = schema_grid(off_out[qi], task_id, qi)
        xg = schema_grid(on_out[qi], task_id, qi)

        pred_off[task_id].append({"attempt_1": og, "attempt_2": og})
        pred_xhigh[task_id].append({"attempt_1": xg, "attempt_2": xg})

        first_grid = xg if best_first == "xhigh" else og
        second_grid = og if best_first == "xhigh" else xg
        pred_union[task_id].append({
            "attempt_1": first_grid,
            "attempt_2": second_grid,
        })

for name, payload in [
    ("predictions_thinking_off.json", pred_off),
    ("predictions_thinking_xhigh.json", pred_xhigh),
    ("predictions_union_off_xhigh.json", pred_union),
]:
    with open(WORK_ROOT / name, "w", encoding="utf-8") as f:
        json.dump(payload, f, separators=(",", ":"))

print("Union attempt_1 mode:", best_first)
print("Union attempt_2 mode:", other)


Union attempt_1 mode: xhigh
Union attempt_2 mode: off


In [13]:

# =====================================================================================
# CELL 13 — SELF-CHECK: OFFICIAL-STYLE FILES RE-SCORE TO THE SAME NUMBERS
# =====================================================================================
def score_official_payload(payload):
    rows = []
    for task_id in ACTIVE_TASK_IDS:
        targets = eval_solutions[task_id]
        attempts = payload.get(task_id, [])
        for qi, target in enumerate(targets):
            rec = attempts[qi] if qi < len(attempts) else {}
            a1 = rec.get("attempt_1")
            a2 = rec.get("attempt_2")
            rows.append({
                "task_id": task_id,
                "qi": qi,
                "pass1": int(grids_equal(a1, target)),
                "pass2": int(grids_equal(a1, target) or grids_equal(a2, target)),
                "weight": weight_map[task_id],
            })
    d = pd.DataFrame(rows)
    return {
        "pass1": float(d["pass1"].mean()),
        "pass2": float(d["pass2"].mean()),
        "weighted_pass1": weighted_mean(d["pass1"], d["weight"]),
        "weighted_pass2": weighted_mean(d["pass2"], d["weight"]),
    }

check_off = score_official_payload(pred_off)
check_on = score_official_payload(pred_xhigh)
check_union = score_official_payload(pred_union)

print("Replay score OFF  :", check_off)
print("Replay score XHIGH:", check_on)
print("Replay score UNION:", check_union)

assert abs(check_off["pass1"] - off_acc) < 1e-12
assert abs(check_on["pass1"] - on_acc) < 1e-12
assert abs(check_union["pass2"] - union_summary["raw_accuracy"]) < 1e-12
print("Replay self-check: PASS")


Replay score OFF  : {'pass1': 0.0, 'pass2': 0.0, 'weighted_pass1': 0.0, 'weighted_pass2': 0.0}
Replay score XHIGH: {'pass1': 0.0, 'pass2': 0.0, 'weighted_pass1': 0.0, 'weighted_pass2': 0.0}
Replay score UNION: {'pass1': 0.0, 'pass2': 0.0, 'weighted_pass1': 0.0, 'weighted_pass2': 0.0}
Replay self-check: PASS


In [14]:
# =====================================================================================
# CELL 14 — Q1 METADATA + SOURCE-ALIGNMENT REPORT FOR CHATGPT
# =====================================================================================
total_elapsed_sec = time.perf_counter() - notebook_start

research_basis = {
    "source_notebook_sha256": SOURCE_FILE_SHA256,
    "proxy48_ordered_sha256": PROXY48_ORDERED_SHA256,
    "proxy48_source_counts": proxy_source_counts,
    "nvarc_diagnostic_anchors": NVARC_DIAGNOSTIC_ANCHORS,
    "qwen_model_card": "https://huggingface.co/Qwen/Qwen3.8-27B-FP8",
    "vllm_reasoning": "https://docs.vllm.ai/en/latest/features/reasoning_outputs/",
    "arc_agi_2": "https://arcprize.org/arc-agi/2",
    "note": "External URLs are provenance only; Q1 design is frozen from the supplied source notebooks/notes.",
}

run_metadata = {
    "experiment": EXPERIMENT_ID,
    "run_profile": RUN_PROFILE,
    "task_set": TASK_SET,
    "accuracy_is_real": ACCURACY_IS_REAL,
    "active_task_ids": ACTIVE_TASK_IDS,
    "proxy48_ordered_sha256": PROXY48_ORDERED_SHA256,
    "proxy48_source_counts": proxy_source_counts,
    "run_id": RUN_ID,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "generation_contract_version": GENERATION_CONTRACT_VERSION,
    "parser_contract_version": PARSER_CONTRACT_VERSION,
    "prompt_contract_version": PROMPT_CONTRACT_VERSION,
    "seed": SEED,
    "model_path": str(MODEL_PATH),
    "model_type": model_cfg.get("model_type"),
    "quantization_config": qcfg,
    "gpu_info": gpu_info,
    "vllm_version": RUNTIME_VLLM_VERSION,
    "torch_version": RUNTIME_TORCH_VERSION,
    "transformers_version": RUNTIME_TRANSFORMERS_VERSION,
    "tokenizers_version": RUNTIME_TOKENIZERS_VERSION,
    "huggingface_hub_version": RUNTIME_HF_HUB_VERSION,
    "cuda_version": RUNTIME_CUDA_VERSION,
    "runtime_stack": RUNTIME_STACK,
    "runtime_stack_sha256": RUNTIME_STACK_SHA256,
    "model_artifact_fingerprint_sha256": MODEL_ARTIFACT_FINGERPRINT_SHA256,
    "engine_config_sha256": ENGINE_CONFIG_SHA256,
    "model_load_sec": model_load_sec,
    "warmup_sec": warmup_sec,
    "total_elapsed_sec": total_elapsed_sec,
    "max_model_len": MAX_MODEL_LEN,
    "max_num_batched_tokens": MAX_NUM_BATCHED_TOKENS,
    "max_num_seqs": MAX_NUM_SEQS,
    "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
    "render_modes": render_modes,
    "chat_template_sha256": template_sha,
    "mode_config": MODE_CONFIG,
    "qwen_official_sampling_reference": QWEN_OFFICIAL_SAMPLING_REFERENCE,
    "exp0_propensity": {
        "cv_auc_eval_vs_test": prop_auc,
        "cv_log_loss": prop_logloss,
        "effective_sample_size_eval_tasks": weight_ess,
        "note": "Diagnostic only; it does not replace raw output-weighted Proxy-48 metrics.",
    },
    "mode_summary": mode_summary.to_dict(orient="records"),
    "union_summary": union_summary,
    "strict_task_accuracy": {
        "off": float(task_scores["off_all_queries"].mean()),
        "xhigh": float(task_scores["xhigh_all_queries"].mean()),
        "union": float(task_scores["union_all_queries"].mean()),
    },
    "candidate_registry_rows": int(len(registry_df)),
    "candidate_cluster_rows": int(len(clusters)),
    "run_complete": run_complete,
    "coverage": coverage,
    "checkpoint_audit": checkpoint_audit,
    "union_attempt_1_mode": best_first,
    "union_attempt_2_mode": other,
    "research_basis": research_basis,
}
with open(WORK_ROOT / "run_metadata.json", "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, indent=2, default=str)

compact_summary = {
    "experiment": EXPERIMENT_ID,
    "run_profile": RUN_PROFILE,
    "task_set": TASK_SET,
    "accuracy_is_real": ACCURACY_IS_REAL,
    "run_complete": run_complete,
    "coverage": coverage,
    "checkpoint_audit": checkpoint_audit,
    "off_raw": off_acc,
    "xhigh_raw": on_acc,
    "union_pass2_raw": union_summary["raw_accuracy"],
    "off_weighted_diagnostic": float(mode_summary.loc[mode_summary["mode"]=="off", "test_weighted_accuracy"].iloc[0]),
    "xhigh_weighted_diagnostic": float(mode_summary.loc[mode_summary["mode"]=="xhigh", "test_weighted_accuracy"].iloc[0]),
    "union_weighted_diagnostic": union_summary["test_weighted_accuracy"],
    "off_only": union_summary["off_only_correct_outputs"],
    "xhigh_only": union_summary["xhigh_only_correct_outputs"],
    "both": union_summary["both_correct_outputs"],
    "neither": union_summary["neither_correct_outputs"],
    "attempts_equal_pct": union_summary["attempts_equal_pct"],
    "candidate_registry_rows": int(len(registry_df)),
    "candidate_cluster_rows": int(len(clusters)),
    "total_elapsed_sec": total_elapsed_sec,
}
with open(WORK_ROOT / "SUMMARY.json", "w", encoding="utf-8") as f:
    json.dump(compact_summary, f, indent=2)

def pct(x):
    return f"{100*x:.2f}%"

report = []
report.append("# Q1-DIR-Q27-PROXY48 — Qwen3.8-27B Direct Baseline Results")
report.append("")
report.append(f"Run ID: `{RUN_ID}`")
report.append(f"Run profile: **{RUN_PROFILE}** | Task set: **{TASK_SET}**")
report.append(f"Proxy-48 SHA256: `{PROXY48_ORDERED_SHA256}`")
report.append(f"Accuracy interpretable: **{ACCURACY_IS_REAL}**")
report.append(f"Active tasks: **{len(ACTIVE_TASK_IDS)}**")
report.append(f"Run status: **{'COMPLETE' if run_complete else 'PARTIAL'}**")
report.append("")
report.append("## Primary output-weighted result")
report.append("")
if not ACCURACY_IS_REAL:
    if RUN_PROFILE == "PREFLIGHT":
        report.append("> PREFLIGHT uses mock predictions. Accuracy values are pipeline checks only and MUST NOT be interpreted as Qwen performance.")
    else:
        report.append("> SMOKE_GPU is a runtime-only gate with short decode caps. Its accuracy MUST NOT be interpreted as the Q1 capability result.")
    report.append("")
if not run_complete:
    report.append("> RUN IS PARTIAL: exact compatible active-task coverage is incomplete; scores below are not final.")
    report.append("")
report.append(f"- Thinking OFF raw exact: **{pct(off_acc)}**")
report.append(f"- Thinking XHIGH raw exact: **{pct(on_acc)}**")
report.append(f"- OFF ∪ XHIGH pass@2: **{pct(union_summary['raw_accuracy'])}**")
report.append("")
report.append("## Coverage / complementarity")
report.append("")
report.append(f"- OFF-only correct outputs: **{union_summary['off_only_correct_outputs']}**")
report.append(f"- XHIGH-only correct outputs: **{union_summary['xhigh_only_correct_outputs']}**")
report.append(f"- Both correct: **{union_summary['both_correct_outputs']}**")
report.append(f"- Neither correct: **{union_summary['neither_correct_outputs']}**")
report.append(f"- Identical valid OFF/XHIGH outputs: **{union_summary['attempts_equal_pct']:.2f}%**")
report.append(f"- Candidate registry rows: **{len(registry_df)}** | exact-grid clusters: **{len(clusters)}**")
report.append(f"- Ignored stale/incompatible checkpoint rows: **{checkpoint_audit['stale_or_incompatible_rows']}**")
report.append(f"- Ignored foreign checkpoint rows: **{checkpoint_audit['foreign_rows']}**")
for _mode in ["off", "xhigh"]:
    _cov = coverage[_mode]
    report.append(
        f"- {_mode} exact compatible coverage: **{_cov['compatible_ok_tasks']}/{_cov['expected_tasks']}** "
        f"| missing={len(_cov['missing_tasks'])} | extra_raw={len(_cov['extra_raw_ok_tasks'])}"
    )
report.append("")
report.append("## Interface / compute")
report.append("")
report.append("`explicit-<think> tokens` counts only text inside literal `<think>...</think>` tags. "
              "Qwen3.8 may emit visible untagged reasoning in this vLLM/template path; zero therefore does NOT mean no reasoning.")
for rec in mode_summary.to_dict(orient="records"):
    report.append(
        f"- {rec['mode']}: parse-valid={rec['parse_valid_task_pct']:.2f}%, "
        f"length-limit={rec['length_limit_task_pct']:.2f}%, "
        f"avg generated={rec['avg_generated_tokens']:.1f}, "
        f"avg explicit-<think> tokens={rec['avg_visible_reasoning_tokens']:.1f} tokens"
    )
report.append(f"- Model load: {model_load_sec:.1f}s | total elapsed: {total_elapsed_sec/60:.1f} min")
report.append("")
report.append("## Frozen source context (not current Q1 results)")
report.append("")
report.append("- NVARC diagnostic: GT present in 55/172 output pools (32.0%); current dual 33.50 vs pool oracle 41.17.")
report.append("- NVARC diagnostic: average raw duplicate ratio 37.8%; scorer rank correlation 0.961 on GT-present cases.")
report.append("- Therefore Q1 saves canonical candidates, exact-grid clusters and provenance for later NVARC∪Qwen oracle analysis; it does not alter NVARC.")
report.append("")
report.append("## Promotion gate")
report.append("")
report.append("Promote beyond Q1 only after checking: valid-grid/parse reliability, truncation, OFF-vs-XHIGH unique coverage, source-EDA slice behavior, tokens/wall-clock, and later offline NVARC union/oracle coverage from saved candidates.")
report.append("The next causal comparison after a usable direct branch is PIV-Q27-P0 (stateful REPL) vs PTQ-Q27-T0 (fresh-call deterministic repair) under matched budgets.")
report.append("")
report.append("## Files to upload/analyze")
report.append("")
for name in [
    "raw_generations.jsonl", "current_generation_records.jsonl", "current_compatible_generations.jsonl",
    "checkpoint_audit.json", "checkpoint_contract_selftest.json",
    "candidate_registry.jsonl", "candidate_clusters.jsonl",
    "task_output_scores.csv", "category_scores.csv", "generation_diagnostics.csv",
    "correctness_overlap.json", "predictions_union_off_xhigh.json",
    "proxy48_source_eda.csv", "SOURCE_ALIGNMENT.json", "run_metadata.json"
]:
    report.append(f"- `{name}`")

with open(WORK_ROOT / "REPORT_FOR_CHATGPT.md", "w", encoding="utf-8") as f:
    f.write("\n".join(report))
with open(WORK_ROOT / "README_UPLOAD_THIS.txt", "w", encoding="utf-8") as f:
    f.write(
        "Q1-DIR-Q27-PROXY48\n"
        f"Run profile: {RUN_PROFILE}\n"
        "Upload this bundle back to ChatGPT after Kaggle Run All:\n"
        f"/kaggle/working/{BUNDLE_NAME}\n\n"
        "PREFLIGHT accuracy is synthetic. SMOKE_GPU is runtime-only. PROXY48 is the Q1 research run.\n"
    )
print("\n".join(report[:38]))


# Q1-DIR-Q27-PROXY48 — Qwen3.8-27B Direct Baseline Results

Run ID: `20260913T134749Z`
Run profile: **PROXY48** | Task set: **PROXY48**
Proxy-48 SHA256: `8f432c60a35091dcda2b890574b1f4d3c03aaf48a5b3b7f96b4e88c35d359134`
Accuracy interpretable: **True**
Active tasks: **48**
Run status: **COMPLETE**

## Primary output-weighted result

- Thinking OFF raw exact: **0.00%**
- Thinking XHIGH raw exact: **0.00%**
- OFF ∪ XHIGH pass@2: **0.00%**

## Coverage / complementarity

- OFF-only correct outputs: **0**
- XHIGH-only correct outputs: **0**
- Both correct: **0**
- Neither correct: **56**
- Identical valid OFF/XHIGH outputs: **0.00%**
- Candidate registry rows: **112** | exact-grid clusters: **31**
- Ignored stale/incompatible checkpoint rows: **0**
- Ignored foreign checkpoint rows: **0**
- off exact compatible coverage: **48/48** | missing=0 | extra_raw=0
- xhigh exact compatible coverage: **48/48** | missing=0 | extra_raw=0

## Interface / compute

`explicit-<think> tokens` counts only t

In [15]:

# =====================================================================================
# CELL 15 — MANIFEST + ZIP EVERYTHING FOR THE NEXT ANALYSIS
# =====================================================================================
def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

# Save the latest de-duplicated generation table without giant prompt/completion fields as CSV.
compact_generation_cols = [
    "run_id", "record_id", "task_id", "mode", "status", "seed", "n_train", "n_test",
    "prompt_tokens", "generated_tokens", "reasoning_tokens_visible",
    "answer_tokens_visible", "finish_reason", "stop_reason", "error",
    "hit_length_limit", "parse_valid", "parser", "parser_confidence", "schema",
    "batch_id", "batch_size_actual", "effective_batch_sec_per_task", "render_method",
    "prompt_sha256", "generation_contract_sha256", "sampling_config_sha256",
    "model_artifact_fingerprint_sha256", "runtime_stack_sha256", "engine_config_sha256",
]
compact_generation_cols = [c for c in compact_generation_cols if c in latest.columns]
latest[compact_generation_cols].to_csv(
    WORK_ROOT / "generation_summary.csv", index=False
)

# Store research basis separately.
with open(WORK_ROOT / "research_basis.json", "w", encoding="utf-8") as f:
    json.dump(research_basis, f, indent=2)

# First create manifest for everything currently present.
manifest_rows = []
for p in sorted(WORK_ROOT.iterdir()):
    if p.is_file() and p.name != "MANIFEST.csv":
        manifest_rows.append({
            "file": p.name,
            "bytes": p.stat().st_size,
            "sha256": sha256_file(p),
        })
manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(WORK_ROOT / "MANIFEST.csv", index=False)

ZIP_PATH = Path("/kaggle/working") / BUNDLE_NAME
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(WORK_ROOT.iterdir()):
        if p.is_file():
            zf.write(p, arcname=p.name)

print("=" * 116)
print(f"Q1-DIR-Q27-PROXY48 {RUN_PROFILE} PIPELINE COMPLETE")
print("=" * 116)
print("Bundle:", ZIP_PATH)
print("ZIP size:", round(ZIP_PATH.stat().st_size / 1024**2, 2), "MB")
print("\nUPLOAD THIS ZIP BACK TO CHATGPT:")
print(ZIP_PATH)
print("\nBundle contents:")
for p in sorted(WORK_ROOT.iterdir()):
    if p.is_file():
        print(" -", p.name)


Q1-DIR-Q27-PROXY48 PROXY48 PIPELINE COMPLETE
Bundle: /kaggle/working/Q1_DIR_Q27_PROXY48_BUNDLE.zip
ZIP size: 2.86 MB

UPLOAD THIS ZIP BACK TO CHATGPT:
/kaggle/working/Q1_DIR_Q27_PROXY48_BUNDLE.zip

Bundle contents:
 - MANIFEST.csv
 - PROGRESS.json
 - README_UPLOAD_THIS.txt
 - REPORT_FOR_CHATGPT.md
 - SOURCE_ALIGNMENT.json
 - SUMMARY.json
 - active_task_selection.csv
 - candidate_clusters.jsonl
 - candidate_events.jsonl
 - candidate_registry.csv
 - candidate_registry.jsonl
 - category_scores.csv
 - chat_template_fingerprint.txt
 - checkpoint_audit.json
 - checkpoint_contract_selftest.json
 - context_config.json
 - correctness_overlap.json
 - current_compatible_generations.jsonl
 - current_generation_records.jsonl
 - eval_test_propensity_weights.csv
 - generation_diagnostics.csv
 - generation_summary.csv
 - mode_summary.csv
 - predictions_thinking_off.json
 - predictions_thinking_xhigh.json
 - predictions_union_off_xhigh.json
 - preload_selftest.json
 - prompt_stats.csv
 - proxy48_source

## Safe run sequence — v1.2 hardened

This revision defaults to `RUN_PROFILE="PREFLIGHT"` because resume/coverage/provenance logic and dependency parity were hardened after the SMOKE audit.

### 1. PREFLIGHT — run this once on v1.2

```python
RUN_PROFILE = "PREFLIGHT"
```

No 27B allocation and no vLLM/CUDA import, but Cell 1 **does install/verify the same offline vLLM 0.19.0 dependency stack** used by production before tokenizer rendering.

It must pass:

- frozen Proxy-48 SHA/count/order contract,
- exact runtime-package parity,
- tokenizer/chat-template rendering,
- tolerant parser tests including truncation guard,
- new checkpoint fingerprint/resume self-tests,
- exact-set coverage guard,
- `no_output -> failed` guard,
- candidate registry/provenance generation,
- replay scorer,
- manifest + ZIP creation.

Expected bundle:

`/kaggle/working/Q1_DIR_Q27_PREFLIGHT_BUNDLE.zip`

### 2. PROXY48 — actual Q1 experiment

After PREFLIGHT v1.2 passes, change **only**:

```python
RUN_PROFILE = "PROXY48"
```

Production budgets remain frozen:

- OFF: 4,096 max generation tokens, batch 16
- XHIGH: 12,288 max generation tokens, batch 8
- same TP=4 / vLLM 0.19.0 / 4×L4 engine config
- same minimal direct prompts and Qwen sampling parameters
- contract-safe batch checkpoint/resume

Expected bundle:

`/kaggle/working/Q1_DIR_Q27_PROXY48_BUNDLE.zip`

### SMOKE status

The supplied v1.1 SMOKE_GPU run already passed real 4×L4 TP=4 model load and warmup. v1.2 does **not** alter the LLM engine configuration, prompt, sampling policy, or parser behavior relative to the post-SMOKE v1.1 notebook. Re-run `SMOKE_GPU` only if PREFLIGHT reports environment drift, Kaggle runtime changes, or you want an extra runtime sanity pass.

### Resume semantics

With `RESET_PROXY48_OUTPUTS=False`, only records whose full generation contract fingerprint matches the current task/prompt/model/runtime/sampling/seed are reusable. Stale or foreign rows are ignored and reported; they cannot satisfy coverage. Set `RESET_PROXY48_OUTPUTS=True` only for an intentional clean rerun.
